## **Домашнее задание 4:** *"Tardigrades: from genestealers to space marines"*

<img src="https://i.redd.it/ole6j5476hsx.jpg" alt="Описание" width="1000">

1. Скачиваем файлы аннотации для дальнейшей работы: **augustus.whole.aa** и **augustus.whole.gff**, полученные в результате работы AUGUSTUS

Перейдём в рабочую папку для выполнения этого задания и для удобства будем работать из неё

In [ ]:
cd ./agitd/task4

2. Выполняем подсчёт общего количества белков

In [ ]:
grep -c "^>" augustus.whole.aa 

16435

Видно, что полученное значение укладывается в указанный в задании диапазон (15-20к). Однако нам необходимо будет отобрать лишь наиболее интересные для нас последовательности. Для этого переходим к следующим шагам

3. Создадим локальную базу данных наших белков

Для начала установим BLAST+ в соответствующее окружение

In [ ]:
conda install -c bioconda blast
conda activate blast

Теперь можем создать локальную базу данных из наших белков

In [ ]:
makeblastdb -in augustus.whole.aa -dbtype prot -out tardigrade_db -parse_seqids

`-in:` входной файл с предсказанными белками тихоходки

`-dbtype prot:` указывает, что мы работаем с белковыми  последовательностями 

`-out tardigrade_db:` задает префикс имени для выходных файлов базы 

`-parse_seqids:` помогает правильно распарсить идентификаторы последовательностей

Теперь у нас есть локальная база данных белков тихоходки **(tardigrade_db)**, и это нам будет необходимо для следующего шага, а именно для выравнивания пептидов, полученных с помощью масс-спектрометрии, на эту самую базу

4. Переходим к данным, полученным с помощью масс-спектрометрии

Теперь необходимо найти, каким белкам из нашей базы эти пептиды соответствуют

In [ ]:
blastp -db tardigrade_db -query peptides.fasta -outfmt "6 qseqid sseqid evalue pident" -out peptide_matches.txt -evalue 1000 -word_size 2 -gapopen 11 -gapextend 1

`-db tardigrade_db:` база данных белков

`-query peptides.fasta:` файл с пептидами

`-outfmt "6 qseqid sseqid evalue pident:` qseqid - ID пептида, sseqid - ID белка, evalue — оценка достоверности, pident - процент идентичности 

`-evalue 1000:` так как для коротких пептидов E-value будет высоким, поэтому мы поднимаем порог, чтобы ничего не потерять.

`-word_size 2:` ищем очень короткие точные совпадения (по умолчанию для белков 3).

`-gapopen/-gapextend:` штрафы за разрывы

Мы сохранили совпадения пептидов из масс-спектра с белками из базы данных **tardigrade_db** в новый файл - peptide_matches.txt

5. Теперь подходим к этапу извлечения белков-кандидатов

Для этого воспользуемся такими инструментами, как samtools

Проиндексируем белковые последовательности

In [ ]:
samtools faidx augustus.whole.aa

Извлечём ID белков из результатов BLAST в текстовый файл:

In [ ]:
cut -f2 peptide_matches.txt | sort -u > protein_ids.txt

И, наконец, извлечём сами последовательности белков-кандидатов в отдельный файл

In [ ]:
samtools faidx augustus.whole.aa -r protein_ids.txt -o candidate_proteins.faa

Подсчитаем количество полученных белков-кандидатов, совпавших с белками из augustus.whole.aa

In [ ]:
grep -c "^>" candidate_proteins.faa

3845

Посмотрим, что это за белки (первые 20 значений)

In [ ]:
grep "^>" candidate_proteins.faa | head -20

>g10000.t1

>g10016.t1

>g10021.t1

>g10031.t1

>g10032.t1

>g1003.t1

>g10048.t1

>g1004.t1

>g10050.t1

>g10056.t1

>g10059.t1

>g1005.t1

>g10072.t1

>g10080.t1

>g10082.t1

>g10085.t1

>g10089.t1

>g10092.t1

>g10096.t1

>g10097.t1

Как видно, пока лишь имеем ID белков, но нам не известно ничего об их функциональной принадлежности. Время переходить к аннотации белков-кандидатов

6. Перейдём к функциональному анализу белков-кандидатов

Для этого сначала создадим базу из белков-кандидатов (тех, которые действительно связаны с ДНК по данным масс-спектрометрии), чтобы узнать о функциях, которые они выполняют

In [ ]:
makeblastdb -in candidate_proteins.faa -dbtype prot -out candidates_db

Далее загрузим аннотированную базу белковых последовательностей - **Swiss-Prot**

In [ ]:
wget https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/uniprot_sprot.fasta.gz
gunzip uniprot_sprot.fasta.gz

Запустим BLAST пептидов-кандидатов по базе Swiss-Prot

In [ ]:
blastp -db swissprot_db -query candidate_proteins.faa -outfmt "6 qseqid sseqid evalue pident stitle" -out candidates_vs_swissprot.txt -evalue 1e-5 -num_threads 4 -max_target_seqs 1

`-outfmt "6 qseqid sseqid evalue pident stitle:` - показывает ID белка, ID  из Swiss-Prot, E-value, процент идентичности и описание функции

`-max_target_seqs 1:` - берет только лучшее совпадение для каждого белка

`-evalue 1e-5:` - стандартный порог значимости

В результате работы этой команды получаем таблицу следующего вида (показаны лишь некоторые строки):

| Protein ID | SwissProt ID | E-value | Identity (%) | Description |
|------------|--------------|---------|--------------|-------------|
| g10000.t1 | sp\|P33363\|BGLX_ECOLI | 0.0 | 40.867 | Periplasmic beta-glucosidase OS=Escherichia coli (strain K12) OX=83333 GN=bglX PE=3 SV=2 |
| g10016.t1 | sp\|P15594\|RTP2_TRYBG | 5.41e-15 | 25.131 | Retrotransposable element SLACS 132 kDa protein OS=Trypanosoma brucei gambiense OX=31285 PE=4 SV=1 |
| g10031.t1 | sp\|Q8C2E4\|PTCD1_MOUSE | 1.62e-85 | 30.968 | Pentatricopeptide repeat-containing protein 1, mitochondrial OS=Mus musculus OX=10090 GN=Ptcd1 PE=2 SV=2 |
| g1003.t1 | sp\|Q99N23\|CAH15_MOUSE | 3.39e-45 | 36.957 | Carbonic anhydrase 15 OS=Mus musculus OX=10090 GN=Ca15 PE=1 SV=1 |
| g10048.t1 | sp\|Q5JRX3\|PREP_HUMAN | 0.0 | 47.996 | Presequence protease, mitochondrial OS=Homo sapiens OX=9606 GN=PITRM1 PE=1 SV=3 |
| g1004.t1 | sp\|P24388\|CRHBP_RAT | 9.37e-08 | 28.125 | Corticotropin-releasing factor-binding protein OS=Rattus norvegicus OX=10116 GN=Crhbp PE=2 SV=1 |
| g10056.t1 | sp\|Q8NC67\|NETO2_HUMAN | 1.36e-33 | 26.389 | Neuropilin and tolloid-like protein 2 OS=Homo sapiens OX=9606 GN=NETO2 PE=1 SV=1 |
| g10059.t1 | sp\|P33363\|BGLX_ECOLI | 1.00e-140 | 39.058 | Periplasmic beta-glucosidase OS=Escherichia coli (strain K12) OX=83333 GN=bglX PE=3 SV=2 |
| g10082.t1 | sp\|Q53H47\|SETMR_HUMAN | 1.15e-10 | 25.442 | Histone-lysine N-methyltransferase SETMAR OS=Homo sapiens OX=9606 GN=SETMAR PE=1 SV=2 |
| g10089.t1 | sp\|P0DPW4\|TX11A_ETHRU | 1.79e-07 | 31.111 | U-scoloptoxin(01)-Er1a OS=Ethmostigmus rubripes OX=62613 PE=3 SV=1 |
| g10092.t1 | sp\|Q99700\|ATX2_HUMAN | 5.92e-15 | 27.869 | Ataxin-2 OS=Homo sapiens OX=9606 GN=ATXN2 PE=1 SV=2 |
| g10096.t1 | sp\|P18293\|ANPRA_MOUSE | 8.58e-136 | 29.925 | Atrial natriuretic peptide receptor 1 OS=Mus musculus OX=10090 GN=Npr1 PE=1 SV=2 |
| g10097.t1 | sp\|Q8IYS1\|P20D2_HUMAN | 4.48e-45 | 34.857 | Xaa-Arg dipeptidase OS=Homo sapiens OX=9606 GN=PM20D2 PE=1 SV=2 |
| g10109.t1 | sp\|Q56078\|BGLX_SALTY | 0.0 | 43.302 | Periplasmic beta-glucosidase OS=Salmonella typhimurium (strain LT2 / SGSC1412 / ATCC 700720) OX=99287 GN=bglX PE=3 SV=2 |
| g10115.t1 | sp\|Q920P9\|LY75_MESAU | 6.80e-11 | 21.555 | Lymphocyte antigen 75 OS=Mesocricetus auratus OX=10036 GN=LY75 PE=2 SV=1 |
| g10115.t1 | sp\|Q920P9\|LY75_MESAU | 2.67e-08 | 22.165 | Lymphocyte antigen 75 OS=Mesocricetus auratus OX=10036 GN=LY75 PE=2 SV=1 |
| g10125.t1 | sp\|P13395\|SPTCA_DROME | 0.0 | 59.791 | Spectrin alpha chain OS=Drosophila melanogaster OX=7227 GN=alpha-Spec PE=1 SV=2 |
| g10125.t1 | sp\|P13395\|SPTCA_DROME | 6.44e-133 | 24.397 | Spectrin alpha chain OS=Drosophila melanogaster OX=7227 GN=alpha-Spec PE=1 SV=2 |
| g10125.t1 | sp\|P13395\|SPTCA_DROME | 3.77e-77 | 25.591 | Spectrin alpha chain OS=Drosophila melanogaster OX=7227 GN=alpha-Spec PE=1 SV=2 |
| g10125.t1 | sp\|P13395\|SPTCA_DROME | 8.63e-75 | 30.838 | Spectrin alpha chain OS=Drosophila melanogaster OX=7227 GN=alpha-Spec PE=1 SV=2 |
| g10125.t1 | sp\|P13395\|SPTCA_DROME | 8.67e-68 | 24.187 | Spectrin alpha chain OS=Drosophila melanogaster OX=7227 GN=alpha-Spec PE=1 SV=2 |
| g10125.t1 | sp\|P13395\|SPTCA_DROME | 9.84e-68 | 25.205 | Spectrin alpha chain OS=Drosophila melanogaster OX=7227 GN=alpha-Spec PE=1 SV=2 |
| g10125.t1 | sp\|P13395\|SPTCA_DROME | 6.64e-52 | 30.789 | Spectrin alpha chain OS=Drosophila melanogaster OX=7227 GN=alpha-Spec PE=1 SV=2 |
| g10125.t1 | sp\|P13395\|SPTCA_DROME | 2.91e-48 | 26.437 | Spectrin alpha chain OS=Drosophila melanogaster OX=7227 GN=alpha-Spec PE=1 SV=2 |
| g10127.t1 | sp\|P15144\|AMPN_HUMAN | 2.82e-84 | 27.970 | Aminopeptidase N OS=Homo sapiens OX=9606 GN=ANPEP PE=1 SV=4 |
| g10132.t1 | sp\|P10079\|FBP1_STRPU | 1.36e-101 | 38.488 | Fibropellin-1 OS=Strongylocentrotus purpuratus OX=7668 GN=EGF1 PE=1 SV=2 |
| g10132.t1 | sp\|P10079\|FBP1_STRPU | 6.10e-98 | 37.201 | Fibropellin-1 OS=Strongylocentrotus purpuratus OX=7668 GN=EGF1 PE=1 SV=2 |
| g10132.t1 | sp\|P10079\|FBP1_STRPU | 1.15e-89 | 37.012 | Fibropellin-1 OS=Strongylocentrotus purpuratus OX=7668 GN=EGF1 PE=1 SV=2 |
| g10132.t1 | sp\|P10079\|FBP1_STRPU | 2.47e-85 | 38.086 | Fibropellin-1 OS=Strongylocentrotus purpuratus OX=7668 GN=EGF1 PE=1 SV=2 |
| g10132.t1 | sp\|P10079\|FBP1_STRPU | 1.28e-59 | 38.462 | Fibropellin-1 OS=Strongylocentrotus purpuratus OX=7668 GN=EGF1 PE=1 SV=2 |

Далее из всего этого разнообразия белковых молекул (а их довольно много) нам интересно поискать те, что в теории смогут быть функционально связаны с защитой ДНК, репарацией, ответом на стресс. Для этого попробуем отфильтровать интересующие последовательности из файла candidates_vs_swissprot.txt по ключевым словам в названии белков

In [ ]:
grep -i -E "DNA|histone|chromatin|helicase|polymerase|topoisomerase|repair|nuclease|exonuclease|endonuclease|ligase|damage|methyltransferase|acetyltransferase|heat shock|chaperone|stress" candidates_vs_swissprot.txt > filtered_candidates.txt

Посчитаем общее количество таких интересных для нас белков 

In [ ]:
cut -f1 filtered_candidates.txt | sort -u | wc -l

284

И это у нас вышло аж из 3845 пептидов-кандидатов, которые получены из результатов масс-спектрометрии

В новом файле filtered.candidates.txt получаем довольно объёмную таблицу, содержащую отфильтрованные белки

| Protein ID | SwissProt ID | E-value | Identity (%) | Description |
|------------|--------------|---------|--------------|-------------|
| g10082.t1 | sp\|Q53H47\|SETMR_HUMAN | 1.15e-10 | 25.442 | Histone-lysine N-methyltransferase SETMAR OS=Homo sapiens OX=9606 GN=SETMAR PE=1 SV=2 |
| g1025.t1 | sp\|O75417\|DPOLQ_HUMAN | 0.0 | 47.190 | DNA polymerase theta OS=Homo sapiens OX=9606 GN=POLQ PE=1 SV=2 |
| g10297.t1 | sp\|Q96PB1\|CASD1_HUMAN | 1.71e-17 | 25.271 | N-acetylneuraminate (7)9-O-acetyltransferase OS=Homo sapiens OX=9606 GN=CASD1 PE=1 SV=1 |
| g10323.t1 | sp\|Q15386\|UBE3C_HUMAN | 3.85e-152 | 41.802 | Ubiquitin-protein ligase E3C OS=Homo sapiens OX=9606 GN=UBE3C PE=1 SV=3 |
| g10325.t1 | sp\|Q5XFW6\|WDR6_RAT | 9.93e-46 | 26.386 | tRNA (34-2'-O)-methyltransferase regulator WDR6 OS=Rattus norvegicus OX=10116 GN=Wdr6 PE=1 SV=2 |
| g10348.t1 | sp\|Q86YW9\|MD12L_HUMAN | 1.96e-131 | 30.036 | Mediator of RNA polymerase II transcription subunit 12-like protein OS=Homo sapiens OX=9606 GN=MED12L PE=1 SV=2 |
| g10349.t1 | sp\|Q8BQM9\|MD12L_MOUSE | 4.31e-87 | 31.241 | Mediator of RNA polymerase II transcription subunit 12-like protein OS=Mus musculus OX=10090 GN=Med12l PE=1 SV=2 |
| g10492.t1 | sp\|Q91ZY8\|TRIM9_RAT | 6.73e-109 | 43.052 | E3 ubiquitin-protein ligase TRIM9 OS=Rattus norvegicus OX=10116 GN=Trim9 PE=1 SV=1 |
| g10574.t1 | sp\|Q9W107\|SYYM_DROME | 6.24e-101 | 41.822 | Tyrosine--tRNA ligase, mitochondrial OS=Drosophila melanogaster OX=7227 GN=TyrRS-m PE=2 SV=1 |
| g10641.t1 | sp\|B8JKD7\|RPA2_DANRE | 1.67e-153 | 37.733 | DNA-directed RNA polymerase I subunit RPA2 OS=Danio rerio OX=7955 GN=polr1b PE=3 SV=1 |
| g10653.t1 | sp\|Q9VGY5\|MET15_DROME | 1.97e-88 | 43.478 | Probable methyltransferase-like protein 15 homolog OS=Drosophila melanogaster OX=7227 GN=CG14683 PE=2 SV=2 |
| g10676.t1 | sp\|P34529\|DCR1_CAEEL | 4.60e-144 | 28.626 | Endoribonuclease dcr-1 OS=Caenorhabditis elegans OX=6239 GN=dcr-1 PE=1 SV=3 |
| g10736.t1 | sp\|Q6TV19\|DICER_DANRE | 2.61e-151 | 28.225 | Endoribonuclease Dicer OS=Danio rerio OX=7955 GN=dicer1 PE=2 SV=2 |
| g10777.t1 | sp\|Q0P4F7\|ACSF2_DANRE | 8.63e-140 | 41.594 | Medium-chain acyl-CoA ligase ACSF2, mitochondrial OS=Danio rerio OX=7955 GN=acsf2 PE=2 SV=1 |
| g10796.t1 | sp\|B3M383\|SPNE_DROAN | 1.01e-21 | 31.646 | Probable ATP-dependent RNA helicase spindle-E OS=Drosophila ananassae OX=7217 GN=spn-E PE=3 SV=1 |
| g10840.t1 | sp\|P62489\|RPB7_RAT | 1.04e-103 | 78.488 | DNA-directed RNA polymerase II subunit RPB7 OS=Rattus norvegicus OX=10116 GN=Polr2g PE=2 SV=1 |
| g10855.t1 | sp\|Q3SZY9\|MED6_BOVIN | 3.38e-62 | 48.077 | Mediator of RNA polymerase II transcription subunit 6 OS=Bos taurus OX=9913 GN=MED6 PE=2 SV=1 |
| g10949.t1 | sp\|Q29S22\|DDX47_BOVIN | 2.23e-150 | 52.391 | Probable ATP-dependent RNA helicase DDX47 OS=Bos taurus OX=9913 GN=DDX47 PE=2 SV=1 |
| g11102.t1 | sp\|Q9H5Z1\|DHX35_HUMAN | 0.0 | 57.279 | Probable ATP-dependent RNA helicase DHX35 OS=Homo sapiens OX=9606 GN=DHX35 PE=1 SV=2 |
| g11171.t1 | sp\|Q4G056\|TYDP1_RAT | 8.42e-125 | 51.090 | Tyrosyl-DNA phosphodiesterase 1 OS=Rattus norvegicus OX=10116 GN=Tdp1 PE=1 SV=1 |
| g11384.t1 | sp\|Q92076\|DPOG1_CHICK | 0.0 | 55.070 | DNA polymerase subunit gamma-1 (Fragment) OS=Gallus gallus OX=9031 GN=POLG PE=2 SV=1 |
| g11427.t1 | sp\|P46935\|NEDD4_MOUSE | 0.0 | 49.754 | E3 ubiquitin-protein ligase NEDD4 OS=Mus musculus OX=10090 GN=Nedd4 PE=1 SV=3 |
| g11432.t1 | sp\|O35286\|DHX15_MOUSE | 0.0 | 77.143 | ATP-dependent RNA helicase DHX15 OS=Mus musculus OX=10090 GN=Dhx15 PE=1 SV=2 |
| g11445.t1 | sp\|P28715\|ERCC5_HUMAN | 2.40e-75 | 40.401 | DNA excision repair protein ERCC-5 OS=Homo sapiens OX=9606 GN=ERCC5 PE=1 SV=3 |
| g11532.t1 | sp\|P97494\|GSH1_MOUSE | 0.0 | 58.346 | Glutamate--cysteine ligase catalytic subunit OS=Mus musculus OX=10090 GN=Gclc PE=1 SV=4 |
| g11557.t1 | sp\|Q9ULU4\|ZMYD8_HUMAN | 1.21e-24 | 27.744 | MYND-type zinc finger-containing chromatin reader ZMYND8 OS=Homo sapiens OX=9606 GN=ZMYND8 PE=1 SV=2 |
| g11569.t1 | sp\|Q18212\|DX39B_CAEEL | 3.07e-153 | 56.701 | Spliceosome RNA helicase DDX39B homolog OS=Caenorhabditis elegans OX=6239 GN=hel-1 PE=1 SV=1 |
| g11630.t1 | sp\|O82175\|SUVH5_ARATH | 1.39e-18 | 31.250 | Histone-lysine N-methyltransferase, H3 lysine-9 specific SUVH5 OS=Arabidopsis thaliana OX=3702 GN=SUVH5 PE=1 SV=1 |
| g11632.t1 | sp\|Q5DW34\|EHMT1_MOUSE | 1.46e-25 | 25.822 | Histone-lysine N-methyltransferase EHMT1 OS=Mus musculus OX=10090 GN=Ehmt1 PE=1 SV=2 |
| g11658.t1 | sp\|Q4R5M3\|ANM5_MACFA | 3.06e-88 | 38.197 | Protein arginine N-methyltransferase 5 OS=Macaca fascicularis OX=9541 GN=PRMT5 PE=2 SV=3 |
| g1168.t1 | sp\|Q4KMD7\|STPAP_DANRE | 5.57e-33 | 29.268 | Speckle targeted PIP5K1A-regulated poly(A) polymerase OS=Danio rerio OX=7955 GN=tut1 PE=2 SV=1 |
| g11715.t1 | sp\|Q6P6Z0\|DDB1_XENLA | 1.53e-88 | 24.169 | DNA damage-binding protein 1 OS=Xenopus laevis OX=8355 GN=ddb1 PE=2 SV=1 |
| g11721.t1 | sp\|B0BN95\|HARB1_RAT | 8.22e-06 | 25.517 | Putative nuclease HARBI1 OS=Rattus norvegicus OX=10116 GN=Harbi1 PE=2 SV=1 |
| g11724.t1 | sp\|Q9VH20\|TAF1B_DROME | 5.01e-07 | 29.078 | TATA box-binding protein-associated factor RNA polymerase I subunit B OS=Drosophila melanogaster OX=7227 GN=TAF1B PE=1 SV=1 |
| g11810.t1 | sp\|Q9W6K1\|MRE11_XENLA | 1.75e-99 | 43.990 | Double-strand break repair protein MRE11 OS=Xenopus laevis OX=8355 GN=mre11 PE=1 SV=1 |
| g11929.t1 | sp\|Q9D7M8\|RPB4_MOUSE | 1.08e-44 | 60.630 | DNA-directed RNA polymerase II subunit RPB4 OS=Mus musculus OX=10090 GN=Polr2d PE=1 SV=2 |
| g11960.t1 | sp\|Q8CJB9\|BRE1B_RAT | 7.00e-98 | 26.956 | E3 ubiquitin-protein ligase BRE1B OS=Rattus norvegicus OX=10116 GN=Rnf40 PE=1 SV=1 |
| g12016.t1 | sp\|A1ZBT5\|MED8_DROME | 2.92e-32 | 38.710 | Mediator of RNA polymerase II transcription subunit 8 OS=Drosophila melanogaster OX=7227 GN=MED8 PE=1 SV=1 |
| g12063.t1 | sp\|Q9NEL2\|SSL1_CAEEL | 4.51e-165 | 44.477 | Helicase ssl-1 OS=Caenorhabditis elegans OX=6239 GN=ssl-1 PE=2 SV=4 |
| g12121.t1 | sp\|Q53H47\|SETMR_HUMAN | 4.81e-10 | 25.088 | Histone-lysine N-methyltransferase SETMAR OS=Homo sapiens OX=9606 GN=SETMAR PE=1 SV=2 |
| g12128.t1 | sp\|Q54QS3\|DDX3_DICDI | 2.70e-59 | 34.694 | Probable ATP-dependent RNA helicase ddx3 OS=Dictyostelium discoideum OX=44689 GN=ddx3 PE=3 SV=1 |
| g12184.t1 | sp\|P52701\|MSH6_HUMAN | 0.0 | 41.388 | DNA mismatch repair protein Msh6 OS=Homo sapiens OX=9606 GN=MSH6 PE=1 SV=2 |
| g12193.t1 | sp\|Q8NDG6\|TDRD9_HUMAN | 1.27e-83 | 28.203 | ATP-dependent RNA helicase TDRD9 OS=Homo sapiens OX=9606 GN=TDRD9 PE=1 SV=3 |
| g12194.t1 | sp\|Q296Q5\|SPNE_DROPS | 3.05e-46 | 33.775 | Probable ATP-dependent RNA helicase spindle-E OS=Drosophila pseudoobscura pseudoobscura OX=46245 GN=spn-E PE=3 SV=2 |
| g12325.t1 | sp\|A8PV03\|SLX1_BRUMA | 3.51e-64 | 42.366 | Structure-specific endonuclease subunit SLX1 homolog OS=Brugia malayi OX=6279 GN=Bm1_35165 PE=3 SV=1 |
| g12416.t1 | sp\|Q53H47\|SETMR_HUMAN | 1.83e-11 | 25.795 | Histone-lysine N-methyltransferase SETMAR OS=Homo sapiens OX=9606 GN=SETMAR PE=1 SV=2 |
| g1264.t1 | sp\|Q5DW34\|EHMT1_MOUSE | 1.27e-15 | 23.033 | Histone-lysine N-methyltransferase EHMT1 OS=Mus musculus OX=10090 GN=Ehmt1 PE=1 SV=2 |
| g12660.t1 | sp\|Q9Y236\|OSGI2_HUMAN | 3.22e-75 | 33.779 | Oxidative stress-induced growth inhibitor 2 OS=Homo sapiens OX=9606 GN=OSGIN2 PE=1 SV=1 |
| g12742.t1 | sp\|Q9Y0H4\|SUDX_DROME | 0.0 | 47.692 | E3 ubiquitin-protein ligase Su(dx) OS=Drosophila melanogaster OX=7227 GN=Su(dx) PE=1 SV=1 |
| g12860.t1 | sp\|Q8WQG9\|JNK1_CAEEL | 2.17e-06 | 42.466 | Stress-activated protein kinase jnk-1 OS=Caenorhabditis elegans OX=6239 GN=jnk-1 PE=1 SV=2 |
| g12883.t1 | sp\|Q6PBT5\|PYRD1_DANRE | 3.50e-57 | 39.322 | tRNA ligase complex-associated NAD(P)H dehydrogenase PYROXD1 OS=Danio rerio OX=7955 GN=pyroxd1 PE=2 SV=1 |
| g1292.t1 | sp\|Q91WN1\|DNJC9_MOUSE | 3.23e-53 | 40.076 | DnaJ homolog subfamily C member 9 OS=Mus musculus OX=10090 GN=Dnajc9 PE=1 SV=2 |
| g12972.t1 | sp\|Q14181\|DPOA2_HUMAN | 1.42e-80 | 30.063 | DNA polymerase alpha subunit B OS=Homo sapiens OX=9606 GN=POLA2 PE=1 SV=2 |
| g13018.t1 | sp\|Q9BSV6\|SEN34_HUMAN | 3.83e-19 | 25.857 | tRNA-splicing endonuclease subunit Sen34 OS=Homo sapiens OX=9606 GN=TSEN34 PE=1 SV=1 |
| g13042.t1 | sp\|A8C750\|THADA_CANLF | 5.24e-21 | 27.394 | tRNA (32-2'-O)-methyltransferase regulator THADA OS=Canis lupus familiaris OX=9615 GN=THADA PE=2 SV=1 |
| g13117.t1 | sp\|Q9ES34\|UBE3B_MOUSE | 0.0 | 41.353 | Ubiquitin-protein ligase E3B OS=Mus musculus OX=10090 GN=Ube3b PE=1 SV=3 |
| g13177.t1 | sp\|Q9H9B1\|EHMT1_HUMAN | 3.33e-49 | 26.667 | Histone-lysine N-methyltransferase EHMT1 OS=Homo sapiens OX=9606 GN=EHMT1 PE=1 SV=4 |
| g13269.t1 | sp\|P19375\|H1E_STRPU | 8.14e-16 | 46.809 | Histone H1, early embryonic OS=Strongylocentrotus purpuratus OX=7668 PE=2 SV=1 |
| g13403.t1 | sp\|Q767K6\|DHX16_PIG | 0.0 | 61.816 | Pre-mRNA-splicing factor ATP-dependent RNA helicase DHX16 OS=Sus scrofa OX=9823 GN=DHX16 PE=3 SV=1 |
| g13489.t1 | sp\|Q6P773\|TAF1C_RAT | 1.09e-14 | 24.181 | TATA box-binding protein-associated factor, RNA polymerase I, subunit C OS=Rattus norvegicus OX=10116 GN=Taf1c PE=2 SV=1 |
| g13540.t1 | sp\|Q04832\|HEXP_LEIMA | 1.24e-09 | 27.815 | DNA-binding protein HEXBP OS=Leishmania major OX=5664 GN=HEXBP PE=4 SV=1 |
| g13571.t1 | sp\|Q05B79\|DHX36_BOVIN | 0.0 | 43.147 | ATP-dependent DNA/RNA helicase DHX36 OS=Bos taurus OX=9913 GN=DHX36 PE=1 SV=1 |
| g13635.t1 | sp\|Q96MB7\|HARB1_HUMAN | 6.93e-07 | 23.669 | Putative nuclease HARBI1 OS=Homo sapiens OX=9606 GN=HARBI1 PE=1 SV=1 |
| g13696.t1 | sp\|Q80VY9\|DHX33_MOUSE | 0.0 | 45.171 | ATP-dependent RNA helicase DHX33 OS=Mus musculus OX=10090 GN=Dhx33 PE=1 SV=1 |
| g13775.t1 | sp\|Q9NV58\|RN19A_HUMAN | 7.32e-144 | 52.284 | E3 ubiquitin-protein ligase RNF19A OS=Homo sapiens OX=9606 GN=RNF19A PE=1 SV=3 |
| g13868.t1 | sp\|P46718\|PDCD2_MOUSE | 2.63e-81 | 39.167 | uS5 assembly chaperone PDCD2 OS=Mus musculus OX=10090 GN=Pdcd2 PE=2 SV=3 |
| g13911.t1 | sp\|Q7K175\|HENMT_DROME | 3.77e-42 | 35.448 | Small RNA 2'-O-methyltransferase OS=Drosophila melanogaster OX=7227 GN=Hen1 PE=1 SV=1 |
| g14015.t1 | sp\|Q28E61\|NSUN2_XENTR | 1.13e-145 | 36.215 | RNA cytosine-C(5)-methyltransferase NSUN2 OS=Xenopus tropicalis OX=8364 GN=nsun2 PE=2 SV=1 |
| g14206.t1 | sp\|Q2KI84\|HARS1_BOVIN | 2.26e-177 | 51.056 | Histidine--tRNA ligase, cytoplasmic OS=Bos taurus OX=9913 GN=HARS1 PE=2 SV=1 |
| g14252.t1 | sp\|Q94K49\|ALP1_ARATH | 6.47e-19 | 24.422 | Protein ANTAGONIST OF LIKE HETEROCHROMATIN PROTEIN 1 OS=Arabidopsis thaliana OX=3702 GN=ALP1 PE=1 SV=1 |
| g14271.t1 | sp\|Q9CWR2\|SMYD3_MOUSE | 2.28e-35 | 35.088 | Histone-lysine N-methyltransferase SMYD3 OS=Mus musculus OX=10090 GN=Smyd3 PE=2 SV=1 |
| g14279.t1 | sp\|Q9Y253\|POLH_HUMAN | 2.94e-92 | 40.260 | DNA polymerase eta OS=Homo sapiens OX=9606 GN=POLH PE=1 SV=1 |
| g14353.t1 | sp\|Q8AV13\|ANM1A_XENLA | 2.06e-120 | 50.543 | Protein arginine N-methyltransferase 1-A OS=Xenopus laevis OX=8355 GN=prmt1-a PE=1 SV=1 |
| g14472.t1 | sp\|P0DOW4\|DSUP_RAMVA | 0.0 | 100.000 | Damage suppressor protein OS=Ramazzottius varieornatus OX=947166 GN=Dsup PE=1 SV=1 |
| g14479.t1 | sp\|Q03112\|MECOM_HUMAN | 9.68e-52 | 66.197 | Histone-lysine N-methyltransferase MECOM OS=Homo sapiens OX=9606 GN=MECOM PE=1 SV=3 |
| g14532.t1 | sp\|P54358\|DPOD1_DROME | 0.0 | 56.218 | DNA polymerase delta catalytic subunit OS=Drosophila melanogaster OX=7227 GN=PolD1 PE=1 SV=2 |
| g14703.t1 | sp\|Q5ZHV2\|NAA35_CHICK | 3.91e-75 | 28.665 | N-alpha-acetyltransferase 35, NatC auxiliary subunit OS=Gallus gallus OX=9031 GN=NAA35 PE=2 SV=1 |
| g14739.t1 | sp\|Q06A37\|CHD7_CHICK | 0.0 | 39.660 | ATP-dependent chromatin remodeler CHD7 OS=Gallus gallus OX=9031 GN=CHD7 PE=2 SV=1 |
| g14782.t1 | sp\|Q6NX31\|MCM7_XENTR | 0.0 | 58.933 | DNA replication licensing factor mcm7 OS=Xenopus tropicalis OX=8364 GN=mcm7 PE=2 SV=1 |
| g14826.t1 | sp\|O82175\|SUVH5_ARATH | 1.02e-30 | 29.286 | Histone-lysine N-methyltransferase, H3 lysine-9 specific SUVH5 OS=Arabidopsis thaliana OX=3702 GN=SUVH5 PE=1 SV=1 |
| g14827.t1 | sp\|Q92598\|HS105_HUMAN | 0.0 | 41.845 | Heat shock protein 105 kDa OS=Homo sapiens OX=9606 GN=HSPH1 PE=1 SV=1 |
| g14891.t1 | sp\|Q5U4E8\|ANM7_RAT | 6.79e-147 | 38.387 | Protein arginine N-methyltransferase 7 OS=Rattus norvegicus OX=10116 GN=Prmt7 PE=2 SV=1 |
| g14936.t1 | sp\|Q08BY1\|MED17_DANRE | 4.76e-45 | 30.935 | Mediator of RNA polymerase II transcription subunit 17 OS=Danio rerio OX=7955 GN=med17 PE=2 SV=1 |
| g14959.t1 | sp\|Q8CFQ3\|AQR_MOUSE | 0.0 | 55.198 | RNA helicase aquarius OS=Mus musculus OX=10090 GN=Aqr PE=1 SV=2 |
| g14969.t1 | sp\|Q4TVV3\|DDX46_DANRE | 0.0 | 57.143 | Probable ATP-dependent RNA helicase DDX46 OS=Danio rerio OX=7955 GN=ddx46 PE=2 SV=1 |
| g14980.t1 | sp\|Q8JZR0\|ACSL5_MOUSE | 0.0 | 45.288 | Long-chain-fatty-acid--CoA ligase 5 OS=Mus musculus OX=10090 GN=Acsl5 PE=1 SV=1 |
| g14998.t1 | sp\|Q9UMR2\|DD19B_HUMAN | 4.44e-123 | 46.465 | ATP-dependent RNA helicase DDX19B OS=Homo sapiens OX=9606 GN=DDX19B PE=1 SV=1 |
| g15015.t1 | sp\|Q68CP4\|HGNAT_HUMAN | 3.21e-128 | 39.483 | Heparan-alpha-glucosaminide N-acetyltransferase OS=Homo sapiens OX=9606 GN=HGSNAT PE=1 SV=2 |
| g15048.t1 | sp\|Q9BWH6\|RPAP1_HUMAN | 1.37e-15 | 26.718 | RNA polymerase II-associated protein 1 OS=Homo sapiens OX=9606 GN=RPAP1 PE=1 SV=3 |
| g15127.t1 | sp\|Q0V9L1\|MMS19_XENTR | 4.21e-37 | 28.218 | MMS19 nucleotide excision repair protein homolog OS=Xenopus tropicalis OX=8364 GN=mms19 PE=2 SV=2 |
| g15275.t1 | sp\|Q14166\|TTL12_HUMAN | 4.40e-148 | 42.295 | Tubulin--tyrosine ligase-like protein 12 OS=Homo sapiens OX=9606 GN=TTLL12 PE=1 SV=2 |
| g15443.t1 | sp\|Q5M7K4\|RBBP4_XENTR | 0.0 | 67.692 | Histone-binding protein RBBP4 OS=Xenopus tropicalis OX=8364 GN=rbbp4 PE=2 SV=3 |
| g15466.t1 | sp\|A7TJM9\|DRS1_VANPO | 6.34e-67 | 34.489 | ATP-dependent RNA helicase DRS1 OS=Vanderwaltozyma polyspora OX=436907 GN=DRS1 PE=3 SV=1 |
| g15521.t1 | sp\|Q99MY8\|ASH1L_MOUSE | 6.31e-60 | 50.000 | Histone-lysine N-methyltransferase ASH1L OS=Mus musculus OX=10090 GN=Ash1l PE=1 SV=3 |
| g15558.t1 | sp\|P18899\|DDR48_YEAST | 3.12e-06 | 34.752 | Stress protein DDR48 OS=Saccharomyces cerevisiae OX=559292 GN=DDR48 PE=1 SV=4 |
| g15649.t1 | sp\|Q5ZJJ2\|RFA1_CHICK | 4.57e-79 | 33.828 | Replication protein A 70 kDa DNA-binding subunit OS=Gallus gallus OX=9031 GN=RPA1 PE=2 SV=1 |
| g15676.t1 | sp\|Q91WA3\|HDA11_MOUSE | 2.58e-138 | 60.671 | Histone deacetylase 11 OS=Mus musculus OX=10090 GN=Hdac11 PE=1 SV=1 |
| g15760.t1 | sp\|Q7ZYC4\|ACBG2_XENLA | 0.0 | 48.071 | Long-chain-fatty-acid--CoA ligase ACSBG2 OS=Xenopus laevis OX=8355 GN=acsbg2 PE=2 SV=1 |
| g15808.t1 | sp\|O14686\|KMT2D_HUMAN | 4.61e-06 | 34.848 | Histone-lysine N-methyltransferase 2D OS=Homo sapiens OX=9606 GN=KMT2D PE=1 SV=2 |
| g15812.t1 | sp\|Q94K49\|ALP1_ARATH | 8.96e-11 | 29.054 | Protein ANTAGONIST OF LIKE HETEROCHROMATIN PROTEIN 1 OS=Arabidopsis thaliana OX=3702 GN=ALP1 PE=1 SV=1 |
| g15826.t1 | sp\|A0JM49\|LTN1_XENTR | 3.83e-63 | 35.174 | E3 ubiquitin-protein ligase listerin OS=Xenopus tropicalis OX=8364 GN=ltn1 PE=2 SV=1 |
| g15867.t1 | sp\|Q7ZZY3\|TOB1B_XENLA | 5.91e-31 | 29.758 | DNA topoisomerase 2-binding protein 1-B OS=Xenopus laevis OX=8355 GN=topbp1-B PE=2 SV=1 |
| g16064.t1 | sp\|Q94K49\|ALP1_ARATH | 1.07e-16 | 23.432 | Protein ANTAGONIST OF LIKE HETEROCHROMATIN PROTEIN 1 OS=Arabidopsis thaliana OX=3702 GN=ALP1 PE=1 SV=1 |
| g16168.t1 | sp\|Q5U538\|HARB1_XENLA | 4.76e-09 | 24.887 | Putative nuclease HARBI1 OS=Xenopus laevis OX=8355 GN=harbi1 PE=2 SV=1 |
| g162.t1 | sp\|Q4R366\|MRPP3_MACFA | 1.51e-65 | 30.722 | Mitochondrial ribonuclease P catalytic subunit OS=Macaca fascicularis OX=9541 GN=PRORP PE=2 SV=2 |
| g16314.t1 | sp\|Q8MNV7\|SMAL1_CAEEL | 5.59e-45 | 56.028 | SNF2 related chromatin remodeling annealing helicase 1 OS=Caenorhabditis elegans OX=6239 GN=smrc-1 PE=3 SV=1 |
| g16404.t1 | sp\|Q94K49\|ALP1_ARATH | 3.13e-18 | 24.092 | Protein ANTAGONIST OF LIKE HETEROCHROMATIN PROTEIN 1 OS=Arabidopsis thaliana OX=3702 GN=ALP1 PE=1 SV=1 |
| g16412.t1 | sp\|Q94K49\|ALP1_ARATH | 2.30e-18 | 24.092 | Protein ANTAGONIST OF LIKE HETEROCHROMATIN PROTEIN 1 OS=Arabidopsis thaliana OX=3702 GN=ALP1 PE=1 SV=1 |
| g16416.t1 | sp\|Q94K49\|ALP1_ARATH | 1.68e-17 | 23.762 | Protein ANTAGONIST OF LIKE HETEROCHROMATIN PROTEIN 1 OS=Arabidopsis thaliana OX=3702 GN=ALP1 PE=1 SV=1 |
| g16417.t1 | sp\|Q94K49\|ALP1_ARATH | 4.60e-17 | 23.432 | Protein ANTAGONIST OF LIKE HETEROCHROMATIN PROTEIN 1 OS=Arabidopsis thaliana OX=3702 GN=ALP1 PE=1 SV=1 |
| g1662.t1 | sp\|Q53H47\|SETMR_HUMAN | 2.30e-11 | 26.866 | Histone-lysine N-methyltransferase SETMAR OS=Homo sapiens OX=9606 GN=SETMAR PE=1 SV=2 |
| g1709.t1 | sp\|B1WB06\|MET24_XENTR | 1.63e-11 | 33.784 | Probable methyltransferase-like protein 24 OS=Xenopus tropicalis OX=8364 GN=mettl24 PE=2 SV=1 |
| g1742.t1 | sp\|Q6PCL9\|PAPOG_MOUSE | 2.21e-152 | 41.368 | Poly(A) polymerase gamma OS=Mus musculus OX=10090 GN=Papolg PE=1 SV=1 |
| g1774.t1 | sp\|Q9P2R7\|SUCB1_HUMAN | 0.0 | 59.395 | Succinate--CoA ligase [ADP-forming] subunit beta, mitochondrial OS=Homo sapiens OX=9606 GN=SUCLA2 PE=1 SV=3 |
| g1818.t1 | sp\|Q7TQ20\|DNJC2_RAT | 3.64e-140 | 42.632 | DnaJ homolog subfamily C member 2 OS=Rattus norvegicus OX=10116 GN=Dnajc2 PE=1 SV=1 |
| g1871.t1 | sp\|Q9H611\|PIF1_HUMAN | 3.33e-11 | 32.544 | ATP-dependent DNA helicase PIF1 OS=Homo sapiens OX=9606 GN=PIF1 PE=1 SV=2 |
| g1892.t1 | sp\|Q94K49\|ALP1_ARATH | 1.76e-17 | 23.762 | Protein ANTAGONIST OF LIKE HETEROCHROMATIN PROTEIN 1 OS=Arabidopsis thaliana OX=3702 GN=ALP1 PE=1 SV=1 |
| g1895.t1 | sp\|A4QP75\|TRM2B_DANRE | 2.47e-99 | 40.796 | tRNA (uracil-5-)-methyltransferase homolog B OS=Danio rerio OX=7955 GN=trmt2b PE=2 SV=2 |
| g189.t1 | sp\|B4F6W9\|TRIPC_XENTR | 7.97e-165 | 35.497 | E3 ubiquitin-protein ligase TRIP12 OS=Xenopus tropicalis OX=8364 GN=trip12 PE=2 SV=1 |
| g2016.t1 | sp\|Q5PQN1\|HERC4_RAT | 0.0 | 41.258 | Probable E3 ubiquitin-protein ligase HERC4 OS=Rattus norvegicus OX=10116 GN=Herc4 PE=2 SV=1 |
| g2038.t1 | sp\|Q6P6W3\|HDAC3_RAT | 0.0 | 63.183 | Histone deacetylase 3 OS=Rattus norvegicus OX=10116 GN=Hdac3 PE=1 SV=1 |
| g2056.t1 | sp\|Q9CZP5\|BCS1_MOUSE | 1.52e-105 | 41.414 | Mitochondrial chaperone BCS1 OS=Mus musculus OX=10090 GN=Bcs1l PE=1 SV=1 |
| g2074.t1 | sp\|O94451\|PLI1_SCHPO | 2.77e-18 | 36.885 | E3 SUMO-protein ligase pli1 OS=Schizosaccharomyces pombe OX=284812 GN=pli1 PE=1 SV=3 |
| g2168.t1 | sp\|Q8QGX4\|PRKDC_CHICK | 0.0 | 27.769 | DNA-dependent protein kinase catalytic subunit OS=Gallus gallus OX=9031 GN=PRKDC PE=2 SV=1 |
| g2278.t1 | sp\|Q9XYZ5\|DDB1_DROME | 9.20e-61 | 27.907

Однако далее, мы отобрали лишь 75 наиболее интересных белков и разделили их в группы по функциональной принадлежности 

**Группа 1.** Белки, участвующие в репарации ДНК

| Protein ID | SwissProt ID | E-value | Identity (%) | Description |
|------------|--------------|---------|--------------|-------------|
| g1025.t1 | sp\|O75417\|DPOLQ_HUMAN | 0.0 | 47.190 | DNA polymerase theta |
| g11445.t1 | sp\|P28715\|ERCC5_HUMAN | 2.40e-75 | 40.401 | DNA excision repair protein ERCC-5 |
| g11715.t1 | sp\|Q6P6Z0\|DDB1_XENLA | 1.53e-88 | 24.169 | DNA damage-binding protein 1 |
| g11810.t1 | sp\|Q9W6K1\|MRE11_XENLA | 1.75e-99 | 43.990 | Double-strand break repair protein MRE11 |
| g12184.t1 | sp\|P52701\|MSH6_HUMAN | 0.0 | 41.388 | DNA mismatch repair protein Msh6 |
| g14472.t1 | sp\|P0DOW4\|DSUP_RAMVA | 0.0 | 100.000 | **Damage suppressor protein** |
| g1871.t1 | sp\|Q9H611\|PIF1_HUMAN | 3.33e-11 | 32.544 | ATP-dependent DNA helicase PIF1 |
| g2168.t1 | sp\|Q8QGX4\|PRKDC_CHICK | 0.0 | 27.769 | DNA-dependent protein kinase catalytic subunit |
| g2278.t1 | sp\|Q9XYZ5\|DDB1_DROME | 9.20e-61 | 27.907 | DNA damage-binding protein 1 |
| g2760.t1 | sp\|Q641W2\|MYG1_RAT | 4.60e-121 | 53.392 | MYG1 exonuclease |
| g4570.t1 | sp\|P51612\|XPC_MOUSE | 1.13e-101 | 33.737 | DNA repair protein complementing XP-C cells homolog |
| g4585.t1 | sp\|P11103\|PARP1_MOUSE | 6.50e-177 | 35.816 | Poly [ADP-ribose] polymerase 1 |
| g518.t1 | sp\|Q03468\|ERCC6_HUMAN | 0.0 | 58.638 | DNA excision repair protein ERCC-6 |
| g5582.t1 | sp\|P24523\|GA45A_CRIGR | 2.51e-08 | 35.165 | Growth arrest and DNA damage-inducible protein GADD45 alpha |
| g7087.t1 | sp\|Q8CFD5\|ERCC8_MOUSE | 2.66e-62 | 38.384 | DNA excision repair protein ERCC-8 |
| g7211.t1 | sp\|P54278\|PMS2_HUMAN | 2.51e-56 | 36.752 | Mismatch repair endonuclease PMS2 |
| g7260.t1 | sp\|Q96RR1\|PEO1_HUMAN | 8.31e-99 | 34.985 | Twinkle mtDNA helicase |
| g7471.t1 | sp\|Q96FI4\|NEIL1_HUMAN | 4.23e-74 | 44.483 | Endonuclease 8-like 1 |
| g7488.t1 | sp\|P97931\|UNG_MOUSE | 1.65e-56 | 48.848 | Uracil-DNA glycosylase |
| g7998.t1 | sp\|Q4G005\|ERCC3_RAT | 0.0 | 69.377 | General transcription and DNA repair factor IIH helicase subunit XPB |
| g8387.t1 | sp\|Q5R6L3\|DNLI4_PONAB | 8.51e-65 | 37.433 | DNA ligase 4 |
| g8658.t1 | sp\|P51612\|XPC_MOUSE | 3.86e-104 | 33.099 | DNA repair protein complementing XP-C cells homolog |
| g9374.t1 | sp\|Q9JK91\|MLH1_MOUSE | 0.0 | 42.517 | DNA mismatch repair protein Mlh1 |

**Группа 2.** Белки-гистоны и модификаторы структуры хроматина

| Protein ID | SwissProt ID | E-value | Identity (%) | Description |
|------------|--------------|---------|--------------|-------------|
| g10082.t1 | sp\|Q53H47\|SETMR_HUMAN | 1.15e-10 | 25.442 | Histone-lysine N-methyltransferase SETMAR |
| g11557.t1 | sp\|Q9ULU4\|ZMYD8_HUMAN | 1.21e-24 | 27.744 | MYND-type zinc finger-containing chromatin reader ZMYND8 |
| g13269.t1 | sp\|P19375\|H1E_STRPU | 8.14e-16 | 46.809 | Histone H1, early embryonic |
| g15443.t1 | sp\|Q5M7K4\|RBBP4_XENTR | 0.0 | 67.692 | Histone-binding protein RBBP4 |
| g15676.t1 | sp\|Q91WA3\|HDA11_MOUSE | 2.58e-138 | 60.671 | Histone deacetylase 11 |
| g2038.t1 | sp\|Q6P6W3\|HDAC3_RAT | 0.0 | 63.183 | Histone deacetylase 3 |
| g391.t1 | sp\|Q02874\|H2AY_RAT | 4.08e-56 | 34.916 | Core histone macro-H2A.1 |
| g8593.t1 | sp\|P16104\|H2AX_HUMAN | 1.37e-47 | 64.754 | Histone H2AX |
| g8602.t1 | sp\|P83038\|HDAC4_CHICK | 0.0 | 62.980 | Histone deacetylase 4 |
| g8850.t1 | sp\|O54941\|SMCE1_MOUSE | 4.63e-38 | 32.283 | SWI/SNF-related matrix-associated actin-dependent regulator of chromatin subfamily E member 1 |
| g9035.t1 | sp\|P56517\|HDAC1_CHICK | 3.87e-117 | 60.294 | Histone deacetylase 1 |
| g9287.t1 | sp\|O60264\|SMCA5_HUMAN | 0.0 | 64.809 | SWI/SNF-related matrix-associated actin-dependent regulator of chromatin subfamily A member 5 |
| g9719.t1 | sp\|P02269\|H2A_ASTRU | 1.38e-45 | 69.919 | Histone H2A |

**Группа 3.** Хеликазы и ДНК-связывающие белки

| Protein ID | SwissProt ID | E-value | Identity (%) | Description |
|------------|--------------|---------|--------------|-------------|
| g10796.t1 | sp\|B3M383\|SPNE_DROAN | 1.01e-21 | 31.646 | Probable ATP-dependent RNA helicase spindle-E |
| g10949.t1 | sp\|Q29S22\|DDX47_BOVIN | 2.23e-150 | 52.391 | Probable ATP-dependent RNA helicase DDX47 |
| g11102.t1 | sp\|Q9H5Z1\|DHX35_HUMAN | 0.0 | 57.279 | Probable ATP-dependent RNA helicase DHX35 |
| g11432.t1 | sp\|O35286\|DHX15_MOUSE | 0.0 | 77.143 | ATP-dependent RNA helicase DHX15 |
| g13571.t1 | sp\|Q05B79\|DHX36_BOVIN | 0.0 | 43.147 | ATP-dependent DNA/RNA helicase DHX36 |
| g14782.t1 | sp\|Q6NX31\|MCM7_XENTR | 0.0 | 58.933 | DNA replication licensing factor mcm7 |
| g3566.t1 | sp\|Q9R050\|SSBP3_RAT | 1.18e-30 | 58.586 | Single-stranded DNA-binding protein 3 |
| g4002.t1 | sp\|O95985\|TOP3B_HUMAN | 0.0 | 49.214 | DNA topoisomerase 3-beta-1 |
| g4324.t1 | sp\|Q5XK83\|MCM4A_XENLA | 0.0 | 53.174 | DNA replication licensing factor mcm4-A |
| g4738.t1 | sp\|Q9Z321\|TOP3B_MOUSE | 6.91e-146 | 44.607 | DNA topoisomerase 3-beta-1 |
| g4789.t1 | sp\|Q498J7\|MC6ZA_XENLA | 0.0 | 55.814 | Zygotic DNA replication licensing factor mcm6-A |
| g7338.t1 | sp\|A2PYH4\|HFM1_HUMAN | 0.0 | 46.989 | Probable ATP-dependent DNA helicase HFM1 |
| g7394.t1 | sp\|P55862\|MCM5A_XENLA | 0.0 | 61.504 | DNA replication licensing factor mcm5-A |

**Группа 4.** Белки, участвующие в ответе на стресс

| Protein ID | SwissProt ID | E-value | Identity (%) | Description |
|------------|--------------|---------|--------------|-------------|
| g14827.t1 | sp\|Q92598\|HS105_HUMAN | 0.0 | 41.845 | Heat shock protein 105 kDa |
| g1818.t1 | sp\|Q7TQ20\|DNJC2_RAT | 3.64e-140 | 42.632 | DnaJ homolog subfamily C member 2 |
| g2056.t1 | sp\|Q9CZP5\|BCS1_MOUSE | 1.52e-105 | 41.414 | Mitochondrial chaperone BCS1 |
| g3515.t1 | sp\|Q3ZCH0\|HSPA9_BOVIN | 0.0 | 79.467 | Stress-70 protein, mitochondrial |
| g5337.t1 | sp\|Q5E9H5\|BCS1_BOVIN | 0.0 | 65.468 | Mitochondrial chaperone BCS1 |
| g7201.t1 | sp\|A5JV83\|HSPBB_DANRE | 1.68e-09 | 25.281 | Heat shock protein beta-11 |
| g7770.t1 | sp\|Q16543\|CDC37_HUMAN | 1.57e-123 | 54.270 | Hsp90 co-chaperone Cdc37 |
| g849.t1 | sp\|Q9JHS4\|CLPX_MOUSE | 0.0 | 63.458 | ATP-dependent clpX-like chaperone, mitochondrial |
| g9727.t1 | sp\|P36604\|BIP_SCHPO | 1.55e-24 | 38.065 | Endoplasmic reticulum chaperone BiP |
| g9728.t1 | sp\|Q24789\|HSP70_ECHGR | 1.30e-10 | 42.105 | Heat shock cognate 70 kDa protein |
| g9736.t1 | sp\|Q5HNW6\|DNAK_STAEQ | 1.68e-20 | 33.136 | Chaperone protein DnaK |

**Группа 5.** Убиквитин-лигазы

| Protein ID | SwissProt ID | E-value | Identity (%) | Description |
|------------|--------------|---------|--------------|-------------|
| g10323.t1 | sp\|Q15386\|UBE3C_HUMAN | 3.85e-152 | 41.802 | Ubiquitin-protein ligase E3C |
| g10492.t1 | sp\|Q91ZY8\|TRIM9_RAT | 6.73e-109 | 43.052 | E3 ubiquitin-protein ligase TRIM9 |
| g11427.t1 | sp\|P46935\|NEDD4_MOUSE | 0.0 | 49.754 | E3 ubiquitin-protein ligase NEDD4 |
| g11960.t1 | sp\|Q8CJB9\|BRE1B_RAT | 7.00e-98 | 26.956 | E3 ubiquitin-protein ligase BRE1B |
| g13117.t1 | sp\|Q9ES34\|UBE3B_MOUSE | 0.0 | 41.353 | Ubiquitin-protein ligase E3B |
| g13775.t1 | sp\|Q9NV58\|RN19A_HUMAN | 7.32e-144 | 52.284 | E3 ubiquitin-protein ligase RNF19A |
| g15826.t1 | sp\|A0JM49\|LTN1_XENTR | 3.83e-63 | 35.174 | E3 ubiquitin-protein ligase listerin |
| g2016.t1 | sp\|Q5PQN1\|HERC4_RAT | 0.0 | 41.258 | Probable E3 ubiquitin-protein ligase HERC4 |
| g5905.t1 | sp\|Q80TP3\|UBR5_MOUSE | 0.0 | 37.966 | E3 ubiquitin-protein ligase UBR5 |
| g6081.t1 | sp\|A2AN08\|UBR4_MOUSE | 0.0 | 32.659 | E3 ubiquitin-protein ligase UBR4 |
| g6630.t1 | sp\|Q7Z6Z7\|HUWE1_HUMAN | 0.0 | 35.528 | E3 ubiquitin-protein ligase HUWE1 |
| g7179.t1 | sp\|Q6WKZ8\|UBR2_MOUSE | 0.0 | 35.410 | E3 ubiquitin-protein ligase UBR2 |
| g8065.t1 | sp\|Q9ULT8\|HECD1_HUMAN | 0.0 | 39.907 | E3 ubiquitin-protein ligase HECTD1 |
| g8742.t1 | sp\|Q149N8\|SHPRH_HUMAN | 5.03e-100 | 27.725 | E3 ubiquitin-protein ligase SHPRH |
| g8882.t1 | sp\|Q86YT6\|MIB1_HUMAN | 0.0 | 68.395 | E3 ubiquitin-protein ligase MIB1 |

Таким образом, мы видим, что полученный список подтверждает исходную гипотезу. Среди кандидатов присутствуют как универсальные белки репарации (MSH6, MLH1, MRE11, ERCC3, PARP1, DNA ligase 4), так и уникальный для тихоходок белок-супрессор повреждений Dsup (100% идентичность), а также множество хеликаз, гистоновых модификаторов и шаперонов. Это объединение протеомных (МС-пептиды) и геномных данных позволяет с высокой достоверностью утверждать, что у *R. varieornatus* с хроматином связана особая сеть белков, ориентированных на детекцию и ликвидацию повреждений ДНК

7. Попробуем установить локализацию полученных белков

Для этого воспользуемся веб-программой **WoLF PSORT**. Переходим по ссылке *https://wolfpsort.hgc.jp/*. Для этого загрузим файл, содержащий аминокислотные последовательности функционально важных белков-кандидатов - *filtered_candidates.txt*. Но так как нам нужны последовательности формата FASTA, для начала преобразуем файл в нужный формат

In [ ]:
cut -f1 filtered_candidates.txt | sort -u > filtered_protein_ids.txt
wc -l filtered_protein_ids.txt

284

Видим, что извлечение нужных белков прошло успешно. Далее используем seqtk (этот инструмент оказался для нас удобнее, чем samtools)

In [ ]:
conda install -c bioconda seqtk
conda activate seqtk

In [ ]:
seqtk subseq candidate_proteins.faa filtered_protein_ids.txt > filtered_candidates.faa

Для работы программы выберем параметры *Animal* и *From Text Area*

Поскольку программа не позволяет загружать файлы свыше 200К, то мы решили загрузить последовательно обе части файла в текстовом виде. Результаты загрузки первой части:

|  |  |  |
|-----------|---------------------|----------------|
| g10082.t1 | cyto | cyto: 18, nucl: 9, extr: 4, plas: 1 |
| g1025.t1 | mito | mito: 12.5, nucl: 11, cyto_mito: 10.1667, cyto_nucl: 9.83333, cyto: 6.5, extr: 1, pero: 1 |
| g10297.t1 | cyto_nucl | cyto_nucl: 12.5, cyto: 11, extr: 6, nucl: 6, plas: 5, mito: 2, pero: 1, lyso: 1 |
| g10323.t1 | nucl | nucl: 12.5, mito: 9.5, cyto_nucl: 8.83333, cyto_mito: 7.33333, plas: 5, cyto: 4, E.R.: 1 |
| g10325.t1 | nucl | nucl: 15.5, cyto_nucl: 13, cyto: 9.5, plas: 6, extr: 1 |
| g10348.t1 | nucl | nucl: 17.5, cyto_nucl: 10.3333, plas: 9, mito: 3.5, cyto_mito: 3.33333, cyto: 2 |
| g10349.t1 | nucl | nucl: 21, cyto_nucl: 14.3333, cyto: 5.5, cyto_mito: 5.16667, mito: 3.5, plas: 2 |
| g10492.t1 | cyto_nucl | cyto_nucl: 13.5, nucl: 12.5, cyto: 11.5, mito: 7, extr: 1 |
| g10574.t1 | mito | mito: 11, plas: 5, E.R.: 5, nucl: 4.5, extr: 4, cyto_nucl: 4, cyto: 2.5 |
| g10641.t1 | cyto | cyto: 16.5, cyto_plas: 9.5, pero: 5, nucl: 4, mito_pero: 4, cysk: 3, plas: 1.5, extr: 1 |
| g10653.t1 | mito | mito: 26.5, E.R._mito: 14, nucl: 3, pero: 2 |
| g10676.t1 | nucl | nucl: 22.5, cyto_nucl: 15, cyto: 6.5, mito: 1, lyso: 1, golg: 1 |
| g10736.t1 | cyto | cyto: 15, nucl: 12, cyto_mito: 9, plas: 2, pero: 2 |
| g10777.t1 | nucl | nucl: 13, plas: 11, cyto_nucl: 9.33333, cyto_mito: 4.16667, cyto: 3.5, mito: 3.5, pero: 1 |
| g10796.t1 | cyto | cyto: 21.5, cyto_nucl: 13.5, mito: 5, extr: 3, nucl: 2.5 |
| g10840.t1 | extr | extr: 26, lyso: 3, plas: 1, cyto: 1, mito: 1 |
| g10855.t1 | nucl | nucl: 18.5, cyto_nucl: 17, cyto: 8.5, extr: 3, pero: 2 |
| g10949.t1 | nucl | nucl: 31, cyto: 1 |
| g11102.t1 | nucl | nucl: 32 |
| g11171.t1 | plas | plas: 30, E.R.: 2 |
| g11384.t1 | nucl | nucl: 12.5, cyto_nucl: 11, cyto: 8.5, mito: 8, E.R.: 2, cysk_plas: 1 |
| g11427.t1 | E.R. | E.R.: 15, plas: 5, extr: 3, nucl: 3, cyto: 2, mito: 2, pero: 1, cysk: 1 |
| g11432.t1 | nucl | nucl: 30.5, cyto_nucl: 16.5, cyto: 1.5 |
| g11445.t1 | nucl | nucl: 27, cyto: 3, E.R.: 1, cysk: 1 |
| g11532.t1 | plas | plas: 14, cyto: 9.5, cyto_nucl: 7, nucl: 3.5, pero: 3, mito_pero: 3, cysk: 1 |
| g11557.t1 | nucl | nucl: 25.5, cyto_nucl: 16, cyto: 5.5, golg: 1 |
| g11569.t1 | mito | mito: 20, cyto: 6.5, cyto_nucl: 5, extr: 3, nucl: 2.5 |
| g11630.t1 | nucl | nucl: 24, plas: 3, cyto: 3, extr: 1, E.R.: 1 |
| g11632.t1 | nucl | nucl: 19.5, cyto_nucl: 14.8333, cyto: 9, cyto_mito: 5.33333, plas: 2, golg: 1 |
| g11658.t1 | nucl | nucl: 17.5, cyto_nucl: 15, cyto: 11.5, pero: 2, cysk: 1 |
| g1168.t1 | plas | plas: 11, cyto: 8, cyto_nucl: 7.5, nucl: 5, E.R.: 4, cysk: 2, mito: 1, pero: 1 |
| g11715.t1 | cyto | cyto: 11.5, cyto_nucl: 9.5, nucl: 6.5, mito: 6, plas: 3, golg: 3, pero: 2 |
| g11721.t1 | nucl | nucl: 21, cyto_nucl: 15, cyto: 7, plas: 2, extr: 1, golg: 1 |
| g11724.t1 | plas | plas: 20, cyto: 5.5, cyto_nucl: 5.5, nucl: 4.5, E.R.: 2 |
| g11810.t1 | cyto_nucl | cyto_nucl: 12.5, nucl: 11.5, cyto: 10.5, pero: 5, extr: 2, mito: 1, E.R.: 1, cysk: 1 |
| g11929.t1 | mito | mito: 28, cyto: 4 |
| g11960.t1 | nucl | nucl: 32 |
| g12016.t1 | mito | mito: 13, pero: 8, cyto_nucl: 5.5, nucl: 5, cyto: 4, plas: 1, extr: 1 |
| g12063.t1 | nucl | nucl: 18.5, cyto_nucl: 16, cyto: 12.5, pero: 1 |
| g12121.t1 | cyto | cyto: 13, nucl: 12, extr: 5, plas: 1, mito: 1 |
| g12128.t1 | nucl | nucl: 16, cyto_nucl: 12.3333, cyto_mito: 7.66667, mito: 7.5, cyto: 6.5, plas: 1, E.R.: 1 |
| g12184.t1 | nucl | nucl: 29, mito: 3 |
| g12193.t1 | nucl | nucl: 30.5, cyto_nucl: 16.5, cyto: 1.5 |
| g12194.t1 | cyto | cyto: 16, extr: 6, nucl: 6, mito: 3, pero: 1 |
| g12325.t1 | cyto | cyto: 17.5, cyto_nucl: 14.5, nucl: 10.5, plas: 2, extr: 1, E.R.: 1 |
| g12416.t1 | nucl | nucl: 15.5, cyto_nucl: 14.5, cyto: 12.5, extr: 4 |
| g1264.t1 | nucl | nucl: 11.5, plas: 11, cyto_nucl: 11, cyto: 7.5, pero: 2 |
| g12660.t1 | nucl | nucl: 12.5, cyto_nucl: 10.5, cyto: 5.5, plas: 4, extr: 4, mito: 4, pero: 2 |
| g12742.t1 | cyto | cyto: 15, cyto_nucl: 14, nucl: 11, golg: 3, mito: 2, extr: 1 |
| g12860.t1 | cyto_nucl | cyto_nucl: 10, cyto: 9, mito: 9, E.R._mito: 7.5, nucl: 5, extr: 3, pero: 3, plas: 1 |
| g12883.t1 | extr | extr: 13, E.R._mito: 5.5, E.R.: 5, mito: 4, cyto_pero: 4, cyto: 3.5, plas: 2, lyso: 2, nucl: 1 |
| g1292.t1 | nucl | nucl: 28, cyto: 3, cysk: 1 |
| g12972.t1 | nucl | nucl: 13, plas: 10, cyto: 6.5, cyto_mito: 4, pero: 1, cysk: 1 |
| g13018.t1 | cyto | cyto: 18.5, cyto_nucl: 16.5, nucl: 11.5, lyso: 1, cysk: 1 |
| g13042.t1 | plas | plas: 18, nucl: 8.5, cyto_nucl: 7, cyto: 4.5, pero: 1 |
| g13117.t1 | nucl | nucl: 16.5, cyto_nucl: 13.8333, cyto: 10, cyto_mito: 5.83333, pero: 5 |
| g13177.t1 | nucl | nucl: 15, cyto_nucl: 12.5, plas: 8, cyto: 8, extr: 1 |
| g13269.t1 | nucl | nucl: 29.5, cyto_nucl: 15.5, cysk: 2 |
| g13403.t1 | nucl | nucl: 27, mito: 5 |
| g13489.t1 | nucl | nucl: 19.5, cyto_nucl: 15.3333, cyto: 10, cyto_mito: 5.83333, pero: 2 |
| g13540.t1 | nucl | nucl: 18, cyto: 13, plas: 1 |
| g13571.t1 | nucl | nucl: 32 |
| g13635.t1 | mito | mito: 17, cyto: 7, plas: 4, golg: 3, E.R.: 1 |
| g13696.t1 | cyto | cyto: 16.5, cyto_nucl: 14.8333, nucl: 11, cyto_plas: 9.16667, pero: 3, cysk: 1 |
| g13775.t1 | plas | plas: 23, nucl: 4.5, cyto_nucl: 4.5, cyto: 3.5, golg: 1 |
| g13868.t1 | cyto | cyto: 17.5, cyto_nucl: 15.5, nucl: 10.5, lyso: 2, extr: 1, pero: 1 |
| g13911.t1 | cyto | cyto: 16, plas: 4, extr: 4, nucl: 4, mito: 2, E.R.: 1, pero: 1 |
| g14015.t1 | cyto | cyto: 16.5, mito: 12, cyto_nucl: 10, nucl: 2.5, cysk: 1 |
| g14206.t1 | cyto | cyto: 31, nucl: 1 |
| g14252.t1 | mito | mito: 23.5, E.R._mito: 14, cyto: 3.5, cyto_nucl: 3, nucl: 1.5, extr: 1, pero: 1 |
| g14271.t1 | nucl | nucl: 27, cyto_nucl: 17, cyto: 5 |
| g14279.t1 | nucl | nucl: 32 |
| g14353.t1 | plas | plas: 11, cyto: 10, mito: 5, nucl: 3, E.R.: 1, pero: 1, cysk: 1 |
| g14472.t1 | nucl | nucl: 28, plas: 2, cyto: 1, cysk: 1 |
| g14479.t1 | nucl | nucl: 32 |
| g14532.t1 | cyto | cyto: 13, cyto_plas: 13, plas: 11.5, cyto_nucl: 10.1667, cyto_mito: 7.5, nucl: 5, extr: 1, golg: 1 |
| g14703.t1 | nucl | nucl: 16.5, cyto_nucl: 12.5, plas: 7, cyto: 3.5, extr: 2, mito: 1, cysk: 1, golg: 1 |
| g14739.t1 | nucl | nucl: 27.5, cyto_nucl: 16.5, cyto: 4.5 |
| g14782.t1 | nucl | nucl: 29, mito: 3 |
| g14826.t1 | nucl | nucl: 17.5, cyto_nucl: 14, cyto: 9.5, extr: 3, plas: 2 |
| g14827.t1 | nucl | nucl: 19.5, cyto_nucl: 14, cyto: 7.5, mito: 4, cysk_plas: 1 |
| g14891.t1 | cyto | cyto: 19, nucl: 5, mito: 3, extr: 2, pero: 2, plas: 1 |
| g14936.t1 | E.R._mito | E.R._mito: 12.5, mito: 9.5, plas: 7, E.R.: 6.5, extr: 3, nucl: 3, cyto_pero: 2.5, cyto: 2 |
| g14959.t1 | cyto | cyto: 16.5, nucl: 11, cyto_mito: 9, pero: 3, cysk: 1 |
| g14969.t1 | nucl | nucl: 32 |
| g14980.t1 | cyto | cyto: 15, pero: 5, mito: 4, nucl: 3, extr: 2, E.R.: 2, plas: 1 |
| g14998.t1 | cyto | cyto: 21, cyto_nucl: 15.3333, cyto_plas: 11.6667, nucl: 4.5, cysk: 4, mito: 1, pero: 1 |
| g15015.t1 | plas | plas: 31, cyto: 1 |
| g15048.t1 | cyto_nucl | cyto_nucl: 12.8333, cyto: 12, nucl: 10.5, cyto_mito: 8, plas: 5, mito: 2.5, E.R.: 2 |
| g15127.t1 | plas | plas: 13.5, cyto_plas: 12.5, cyto: 10.5, nucl: 5, pero: 2, golg: 1 |
| g15275.t1 | cyto | cyto: 11, cyto_nucl: 10, nucl: 7, plas: 6, extr: 5, mito: 1, E.R.: 1, lyso: 1 |
| g15443.t1 | cysk | cysk: 28, cyto: 4 |
| g15466.t1 | nucl | nucl: 32 |
| g15521.t1 | nucl | nucl: 25, cyto_nucl: 17.5, cyto: 6, plas: 1 |
| g15558.t1 | extr | extr: 22, E.R.: 6, lyso: 2, plas: 1, nucl: 1 |
| g15649.t1 | plas | plas: 32 |
| g15676.t1 | cyto | cyto: 16, nucl: 12, mito: 3, golg: 1 |
| g15760.t1 | cyto | cyto: 13, plas: 6, mito: 4.5, mito_pero: 4.5, pero: 3.5, E.R.: 2, golg: 2, nucl: 1 |
| g15808.t1 | nucl | nucl: 23.5, cyto_nucl: 18, cyto: 7.5, pero: 1 |
| g15812.t1 | cyto_nucl | cyto_nucl: 17, cyto: 13.5, nucl: 9.5, extr: 6, plas: 2, pero: 1 |
| g15826.t1 | cyto | cyto: 16.5, cyto_nucl: 12, nucl: 6.5, plas: 6, mito: 2, pero: 1 |
| g15867.t1 | nucl | nucl: 25, cyto_nucl: 15.5, cyto: 4, plas: 1, mito: 1, E.R.: 1 |
| g16064.t1 | mito | mito: 24, E.R._mito: 15, cyto: 2.5, cyto_nucl: 2.5, nucl: 1.5, extr: 1, pero: 1 |
| g16168.t1 | cyto | cyto: 15, nucl: 11, mito: 4, lyso: 1, golg: 1 |
| g162.t1 | mito | mito: 28, cyto_mito: 16.8333, cyto: 3.5, cyto_nucl: 2.66667 |
| g16314.t1 | cyto | cyto: 11, cyto_nucl: 9.5, mito: 7.5, nucl: 6, mito_pero: 5.5, extr: 3, pero: 2.5, plas: 1, E.R.: 1 |
| g16404.t1 | mito | mito: 24, E.R._mito: 15, cyto: 2.5, cyto_nucl: 2.5, nucl: 1.5, extr: 1, pero: 1 |
| g16412.t1 | mito | mito: 20, E.R._mito: 13, nucl: 4, cyto: 4, extr: 1, pero: 1 |
| g16416.t1 | mito | mito: 22, E.R._mito: 14, nucl: 4.5, cyto_nucl: 3.5, cyto: 1.5, extr: 1, pero: 1 |
| g16417.t1 | cysk | cysk: 23, cyto_nucl: 5, cyto: 4.5, nucl: 2.5, mito: 1, lyso: 1 |
| g1662.t1 | nucl | nucl: 12, cyto: 10, extr: 9, plas: 1 |
| g1709.t1 | plas | plas: 11.5, mito: 11.5, extr_plas: 8, cyto_mito: 7.5, extr: 3.5, cyto: 2.5, lyso: 2, golg: 1 |
| g1742.t1 | plas | plas: 11, nucl: 9.5, cyto_nucl: 8.5, cyto: 6.5, E.R.: 4, mito: 1 |
| g1774.t1 | mito | mito: 25.5, cyto_mito: 16.6667, E.R._mito: 13.6667, cyto: 5.5, cyto_nucl: 3.66667 |
| g1818.t1 | nucl | nucl: 32 |
| g1871.t1 | cyto_nucl | cyto_nucl: 16, cyto: 15, nucl: 7, extr: 5, mito: 4, pero: 1 |
| g1892.t1 | mito | mito: 22.5, E.R._mito: 13.5, cyto: 4, nucl: 2, extr: 1, pero: 1 |
| g1895.t1 | mito | mito: 15, nucl: 11, cyto: 6 |
| g189.t1 | nucl | nucl: 23.5, cyto_nucl: 16, cyto: 7.5, golg: 1 |
| g2016.t1 | nucl | nucl: 19, mito: 6, cyto: 4, plas: 2, golg: 1 |
| g2038.t1 | cyto | cyto: 14, plas: 7, nucl: 5, pero: 3, extr: 1, E.R.: 1, golg: 1 |
| g2056.t1 | plas | plas: 31, extr: 1 |
| g2074.t1 | nucl | nucl: 24.5, cyto_nucl: 16.5, cyto: 7.5 |
| g2168.t1 | plas | plas: 15, E.R.: 15, nucl: 1, cyto: 1 |
| g2278.t1 | mito | mito: 14.5, cyto_mito: 9.33333, pero: 6, cyto_nucl: 3.33333, extr: 3, cyto: 3, nucl: 2.5, E.R.: 2, plas: 1 |
| g2331.t1 | nucl | nucl: 12.5, cyto_nucl: 12, cyto: 10.5, pero: 5, cysk: 3, E.R.: 1 |
| g2444.t1 | mito | mito: 11, nucl: 10, extr: 4, cyto: 3, golg: 3, lyso: 1 |
| g2580.t1 | cyto | cyto: 17.5, cyto_mito: 10.3333, cyto_plas: 9.66667, nucl: 7, pero: 4, extr: 1, cysk: 1 |
| g2617.t1 | cyto | cyto: 21.5, cyto_nucl: 15, nucl: 5.5, cysk: 3, mito: 1, lyso: 1 |
| g2755.t1 | nucl | nucl: 23, cyto: 8, cysk: 1 |
| g2756.t1 | nucl | nucl: 19.5, cyto_nucl: 14, cyto: 7.5, extr: 2, golg: 2, plas: 1 |
| g2760.t1 | mito | mito: 19, E.R._mito: 11, cyto_nucl: 7, cyto: 6, nucl: 4, pero: 1, cysk_plas: 1 |
| g2873.t1 | cyto | cyto: 21.5, cyto_nucl: 12.5, extr: 7, nucl: 2.5, mito: 1 |
| g305.t1 | cyto | cyto: 16.5, mito: 11, cyto_nucl: 10, nucl: 2.5, extr: 1, pero: 1 |
| g3355.t1 | nucl | nucl: 16, cyto_nucl: 13.3333, cyto: 8.5, cyto_mito: 5.16667, cysk_plas: 3, plas: 2.5, cysk: 2.5, E.R.: 1, pero: 1 |
| g340.t1 | nucl | nucl: 23.5, cyto_nucl: 16.5, cyto: 8.5 |
| g3515.t1 | nucl | nucl: 15.5, cyto_nucl: 12.8333, cyto: 7, cyto_golg: 4.5, cysk: 3, extr: 2, pero: 2, plas: 1, mito: 1 |
| g3540.t1 | E.R. | E.R.: 30, extr: 2 |
| g3566.t1 | nucl | nucl: 27.5, cyto_nucl: 15.5, cyto: 2.5, mito: 1, cysk_plas: 1 |
| g356.t1 | plas | plas: 25, E.R.: 3, cyto: 2.5, cyto_nucl: 2, pero: 1 |
| g3636.t1 | extr | extr: 26, nucl: 4, cyto: 2 |
| g3737.t1 | cyto | cyto: 20.5, cyto_nucl: 14, nucl: 4.5, extr: 3, mito: 3, cysk: 1 |
| g3813.t1 | cyto | cyto: 12, mito: 9, cyto_pero: 9, cyto_nucl: 8.83333, E.R._mito: 6, pero: 4.5, nucl: 2.5, extr: 2, cysk: 1 |
| g391.t1 | mito | mito: 21, cyto: 8, pero: 2, extr: 1 |
| g3932.t1 | nucl | nucl: 30.5, cyto_nucl: 16.5, cyto: 1.5 |
| g3982.t1 | nucl | nucl: 24.5, cyto_nucl: 17, cyto: 6.5, pero: 1 |
| g4002.t1 | cyto | cyto: 12.5, cyto_nucl: 10, pero: 8.5, nucl: 6.5, mito_pero: 5, extr: 3, plas: 1 |
| g4025.t1 | nucl | nucl: 12.5, cyto_nucl: 12, cyto: 10.5, mito: 5, cysk_plas: 2, plas: 1.5, cysk: 1.5, extr: 1 |
| g4032.t1 | nucl | nucl: 24, cyto_nucl: 14.5, mito: 5, cyto: 3 |
| g4099.t1 | mito | mito: 23, nucl: 3.5, pero: 3, cyto_nucl: 2.5, plas: 2 |
| g410.t1 | extr | extr: 20, nucl: 7, cyto: 5 |
| g411.t1 | nucl | nucl: 19.5, cyto_nucl: 16, cyto: 9.5, plas: 3 |
| g4160.t1 | cyto | cyto: 13.5, cyto_nucl: 13, nucl: 11.5, plas: 6, cysk: 1 |
| g4200.t1 | mito | mito: 11, cyto: 10.5, cyto_nucl: 8.5, nucl: 5.5, pero: 4, extr: 1 |
| g4201.t1 | nucl | nucl: 23.5, cyto_nucl: 14, plas: 4, cyto: 3.5, E.R.: 1 |
| g4324.t1 | nucl | nucl: 22, cyto: 6, plas: 2, E.R.: 1, golg: 1 |
| g4327.t1 | nucl | nucl: 32 |
| g4349.t1 | cyto | cyto: 9.5, extr: 9, cyto_nucl: 7, plas: 4, pero: 3, mito: 2, E.R.: 2, golg: 1 |
| g4361.t1 | nucl | nucl: 15.5, cyto_nucl: 11.5, plas: 11, cyto: 4.5, E.R.: 1 |
| g4504.t1 | nucl | nucl: 22.5, cyto_nucl: 16, cyto: 8.5, golg: 1 |

И вторая часть этого огромного списка:

|  |  |  |
|-----------|---------------------|----------------|
| g4570.t1 | nucl | nucl: 24.5, cyto_nucl: 16, cyto: 6.5, E.R.: 1 |
| g4585.t1 | nucl | nucl: 32 |
| g4593.t1 | nucl | nucl: 25.5, cyto_nucl: 17.5, cyto: 6.5 |
| g4613.t1 | nucl | nucl: 14.5, cyto_nucl: 13.3333, cyto: 9, cyto_pero: 6, extr: 3, mito: 2, golg: 2, pero: 1.5 |
| g4634.t1 | cyto | cyto: 14.5, cyto_nucl: 13, nucl: 8.5, mito: 6, extr: 1, cysk: 1, golg: 1 |
| g4704.t1 | cyto | cyto: 22.5, cyto_nucl: 14.5, extr: 5, nucl: 3.5, lyso: 1 |
| g4738.t1 | nucl | nucl: 30, cyto: 2 |
| g4740.t1 | nucl | nucl: 14.5, cyto_nucl: 12, cyto: 8.5, plas: 8, mito: 1 |
| g4789.t1 | nucl | nucl: 24, cyto: 4, plas: 2, E.R.: 1, golg: 1 |
| g4851.t1 | nucl | nucl: 22, cyto: 5, mito: 5 |
| g4890.t1 | cyto | cyto: 16.5, cyto_nucl: 12, nucl: 6.5, extr: 3, pero: 3, E.R._mito: 2, plas: 1 |
| g494.t1 | mito | mito: 18.5, cyto_mito: 12.8333, cyto_nucl: 5.83333, cyto: 5.5, extr: 5, pero: 1 |
| g5065.t1 | cyto_nucl | cyto_nucl: 14.5, cyto: 13, nucl: 12, extr: 6, mito: 1 |
| g5069.t1 | plas | plas: 10, cyto: 10, pero: 6, nucl: 3, mito: 1, E.R.: 1, cysk: 1 |
| g5104.t1 | nucl | nucl: 13.5, cyto_nucl: 10, plas: 7, cyto: 5.5, mito: 4, pero: 2 |
| g518.t1 | nucl | nucl: 18, cyto_nucl: 12.5, cyto: 5, plas: 3, pero: 3, cysk: 3 |
| g5302.t1 | cyto_nucl | cyto_nucl: 14.3333, nucl: 13.5, cyto: 10, cyto_plas: 8.16667, plas: 4.5, cysk: 3, extr: 1 |
| g5337.t1 | plas | plas: 20, nucl: 9.5, cyto_nucl: 6.33333, cyto: 2, cyto_mito: 1.83333 |
| g5381.t1 | cyto | cyto: 21, cyto_nucl: 13, nucl: 3, mito: 3, golg: 2, E.R.: 1, pero: 1, lyso: 1 |
| g5456.t1 | mito | mito: 24, E.R._mito: 15, cyto: 2.5, cyto_nucl: 2.5, nucl: 1.5, extr: 1, pero: 1 |
| g5582.t1 | extr | extr: 19, nucl: 7, plas: 4, cyto: 2 |
| g5618.t1 | plas | plas: 32 |
| g5721.t1 | nucl | nucl: 20.5, cyto_nucl: 15, cyto: 6.5, plas: 4, mito: 1 |
| g5905.t1 | nucl | nucl: 17.5, cyto_nucl: 14.5, cyto: 8.5, extr: 4, plas: 1, golg: 1 |
| g5927.t1 | nucl | nucl: 30.5, cyto_nucl: 16.5, cyto: 1.5 |
| g5942.t1 | nucl | nucl: 14, cyto_nucl: 12.5, extr: 7, cyto: 7, plas: 4 |
| g5968.t1 | nucl | nucl: 15.5, cyto_nucl: 12.5, plas: 7, cyto: 6.5, pero: 3 |
| g5997.t1 | mito | mito: 17, cyto: 7.5, cyto_nucl: 6.5, nucl: 4.5, pero: 2, plas: 1 |
| g6016.t1 | mito | mito: 15.5, nucl: 11.5, cyto_mito: 10, cyto_nucl: 8.83333, cyto: 3, extr: 2 |
| g6080.t1 | plas | plas: 17, cyto_nucl: 6.33333, cyto: 6, nucl: 5.5, cyto_mito: 3.83333, cysk: 3 |
| g6081.t1 | plas | plas: 7, cyto_nucl: 7, nucl: 6, cyto: 6, mito: 5, extr: 3, E.R.: 2, pero: 2, lyso: 1 |
| g6093.t1 | plas | plas: 27, E.R.: 5 |
| g6118.t1 | nucl | nucl: 24, cyto_nucl: 15, cyto: 4, plas: 2, E.R.: 1, golg: 1 |
| g6125.t1 | nucl | nucl: 11.5, cyto_nucl: 11, cyto: 9.5, mito: 9, cysk: 1.5, cysk_plas: 1.5 |
| g6239.t1 | mito | mito: 22, E.R._mito: 12.5, cyto_nucl: 4, nucl: 3.5, cyto: 3.5, extr: 1, pero: 1 |
| g623.t1 | cyto_nucl | cyto_nucl: 11.5, cyto: 10.5, nucl: 9.5, extr: 8, plas: 2, mito: 1, pero: 1 |
| g6345.t1 | cyto_nucl | cyto_nucl: 9, nucl: 8.5, cyto: 8.5, mito: 8, E.R._mito: 7.5, E.R.: 3, plas: 2, extr: 1, golg: 1 |
| g6351.t1 | nucl | nucl: 29, cyto_nucl: 17, cyto: 3 |
| g6372.t1 | extr | extr: 9, cyto: 8, nucl: 7, plas: 3, mito: 2, golg: 2, pero: 1 |
| g6459.t1 | mito | mito: 14, nucl: 11, cyto_nucl: 10, cyto: 7 |
| g6472.t1 | cyto_nucl | cyto_nucl: 12.5, nucl: 11.5, cyto: 10.5, plas: 10 |
| g651.t1 | cyto | cyto: 19, nucl: 5, E.R.: 3, mito: 2, extr: 1, pero: 1, golg: 1 |
| g6593.t1 | cysk | cysk: 28, cyto: 3, lyso: 1 |
| g6605.t1 | cyto | cyto: 13.5, cyto_nucl: 13, nucl: 11.5, cysk: 3, mito: 2, extr: 1, pero: 1 |
| g6630.t1 | nucl | nucl: 15, plas: 9, cysk: 3, extr: 1, cyto: 1, mito: 1, E.R.: 1, golg: 1 |
| g6657.t1 | nucl | nucl: 25, plas: 3, cyto: 2, pero: 1, golg: 1 |
| g6694.t1 | nucl | nucl: 15.5, cyto_nucl: 14.6667, cyto: 12.5, cyto_mito: 7.83333, pero: 2, cysk: 1 |
| g6698.t1 | nucl | nucl: 14, mito: 13, cyto_nucl: 10, cyto: 4, extr: 1 |
| g673.t1 | nucl | nucl: 13.5, cyto_nucl: 12.5, cyto: 10.5, cysk: 5, cysk_plas: 4, pero: 1, lyso: 1 |
| g6744.t1 | nucl | nucl: 15, cyto: 15, pero: 1, cysk: 1 |
| g6791.t1 | cyto_mito | cyto_mito: 12.6667, mito: 12.5, cyto: 11.5, cyto_nucl: 7.83333, pero: 3, extr: 2, nucl: 2, plas: 1 |
| g691.t1 | nucl | nucl: 13, cyto: 6, plas: 5.5, cysk_plas: 5, cysk: 3.5, pero: 2, mito: 1, E.R.: 1 |
| g6979.t1 | cyto | cyto: 22.5, cyto_nucl: 14, nucl: 4.5, plas: 2, extr: 1, E.R.: 1, pero: 1 |
| g7087.t1 | plas | plas: 11, cyto_nucl: 8.5, nucl: 7.5, cyto: 6.5, mito: 3, lyso: 2, extr: 1, golg: 1 |
| g709.t1 | plas | plas: 15, nucl: 6, E.R.: 5, mito: 3, extr: 2, pero: 1 |
| g7179.t1 | cyto_nucl | cyto_nucl: 10.5, plas: 10, cyto: 9.5, nucl: 8.5, pero: 4 |
| g7201.t1 | cyto | cyto: 19, mito: 10, nucl: 3 |
| g7211.t1 | nucl | nucl: 32 |
| g7224.t1 | cyto_nucl | cyto_nucl: 12, nucl: 11, cyto: 11, plas: 5, extr: 2, mito: 1, pero: 1, golg: 1 |
| g7260.t1 | mito | mito: 27.5, cyto_mito: 15.3333, cyto_nucl: 2.83333, nucl: 2.5, cyto: 2 |
| g7261.t1 | plas | plas: 8, E.R.: 7, cyto: 4.5, cyto_nucl: 4.5, mito: 4, cysk: 4, nucl: 3.5, pero: 1 |
| g7303.t1 | cyto | cyto: 20, cyto_nucl: 16.5, nucl: 9, mito: 2, cysk: 1 |
| g7338.t1 | nucl | nucl: 25, plas: 4, cyto: 2, mito: 1 |
| g7355.t1 | nucl | nucl: 20, cyto: 7, golg: 2, plas: 1, E.R.: 1, pero: 1 |
| g7377.t1 | plas | plas: 28, cyto: 3.5, cyto_nucl: 2.5 |
| g737.t1 | cyto | cyto: 15, nucl: 8, cysk: 4, extr: 2, pero: 2, mito: 1 |
| g7394.t1 | nucl | nucl: 26, cyto: 5, plas: 1 |
| g7454.t1 | cyto | cyto: 17, cyto_nucl: 15.5, nucl: 10, cysk: 2, extr: 1, lyso: 1, golg: 1 |
| g7471.t1 | nucl | nucl: 22, cyto_nucl: 17.1667, cyto: 8, cyto_golg: 5.83333, extr: 1 |
| g7488.t1 | cyto | cyto: 13.5, cyto_nucl: 9, mito: 7, nucl: 3.5, plas: 2, extr: 2, pero: 2, E.R.: 1, golg: 1 |
| g7517.t1 | plas | plas: 25, cyto: 5, mito_pero: 2 |
| g760.t1 | nucl | nucl: 19, cyto_nucl: 15.5, cyto: 10, plas: 3 |
| g7637.t1 | nucl | nucl: 28, mito: 4 |
| g7744.t1 | cyto | cyto: 15.5, cyto_nucl: 10.8333, cyto_mito: 8.66667, pero: 5, nucl: 4, E.R.: 4, plas: 3 |
| g7761.t1 | cyto | cyto: 17.5, cyto_nucl: 14.5, nucl: 8.5, cysk: 3, extr: 1, E.R.: 1, lyso: 1 |
| g7770.t1 | cyto | cyto: 20.5, cyto_nucl: 16, nucl: 10.5, extr: 1 |
| g7861.t1 | nucl | nucl: 16, cyto_nucl: 14, cyto: 8, plas: 5, pero: 1, cysk: 1, golg: 1 |
| g7928.t1 | nucl | nucl: 28, cyto: 4 |
| g7935.t1 | plas | plas: 12, E.R.: 9.5, extr: 9, E.R._golg: 6, golg: 1.5 |
| g7961.t1 | cyto | cyto: 23.5, cyto_nucl: 14.5, nucl: 4.5, mito: 3, extr: 1 |
| g7998.t1 | nucl | nucl: 18.5, cyto_nucl: 12.5, plas: 8, cyto: 5.5 |
| g8013.t1 | nucl | nucl: 22, cyto: 9, golg: 1 |
| g8026.t1 | plas | plas: 32 |
| g8065.t1 | nucl | nucl: 21.5, cyto_nucl: 14.5, plas: 6, cyto: 4.5 |
| g809.t1 | mito | mito: 19, nucl: 4, cyto: 4, pero: 4, lyso: 1 |
| g8130.t1 | plas | plas: 31, E.R.: 1 |
| g8150.t1 | nucl | nucl: 20.5, cyto_nucl: 16, cyto: 8.5, plas: 2, extr: 1 |
| g832.t1 | cyto | cyto: 13, E.R.: 6, nucl: 5, plas: 3, mito: 3, pero: 2 |
| g833.t1 | mito | mito: 19.5, E.R._mito: 10.5, cyto: 5.5, cyto_nucl: 4.5, pero: 3, nucl: 2.5, extr: 1 |
| g8350.t1 | cyto | cyto: 13.5, cyto_nucl: 13, nucl: 11.5, plas: 3, golg: 3, cysk: 1 |
| g8387.t1 | cyto | cyto: 27, nucl: 4, extr: 1 |
| g849.t1 | nucl | nucl: 24, cyto_nucl: 14, mito: 6, cyto: 2 |
| g8593.t1 | plas | plas: 21, nucl: 5.5, cyto_nucl: 4, E.R.: 2, pero: 2, cyto: 1.5 |
| g8602.t1 | cyto_nucl | cyto_nucl: 12, nucl: 11.5, cyto: 9.5, plas: 6, mito: 2, E.R.: 1, pero: 1, golg: 1 |
| g8658.t1 | nucl | nucl: 26, cyto_nucl: 15.5, cyto: 3, mito: 1, E.R.: 1, pero: 1 |
| g8660.t1 | nucl | nucl: 16.5, cyto_nucl: 12.5, cyto: 7.5, plas: 3, mito: 3, pero: 2 |
| g8742.t1 | nucl | nucl: 32 |
| g8771.t1 | nucl | nucl: 11.5, cyto_nucl: 11, cyto: 9.5, plas: 6, mito: 2, E.R.: 2, extr: 1 |
| g8850.t1 | nucl | nucl: 29.5, cyto_nucl: 16.5, cyto: 2.5 |
| g8854.t1 | cyto | cyto: 18.5, cyto_plas: 13, plas: 6.5, nucl: 4, mito_pero: 2, extr: 1 |
| g8882.t1 | nucl | nucl: 27, cyto_nucl: 19, cyto: 5 |
| g8905.t1 | nucl | nucl: 32 |
| g8953.t1 | nucl | nucl: 28, mito: 4 |
| g9035.t1 | nucl | nucl: 17, cyto_nucl: 15.5, cyto: 10, plas: 2.5, cysk_plas: 2, mito: 1, golg: 1 |
| g9081.t1 | mito | mito: 19.5, E.R._mito: 12, cyto: 6.5, cyto_nucl: 4.5, pero: 2, nucl: 1.5, extr: 1 |
| g9097.t1 | cyto | cyto: 10.5, cyto_nucl: 10.5, mito: 8, extr: 7, plas: 2, pero: 1 |
| g9168.t1 | cyto_nucl | cyto_nucl: 12.5, nucl: 11, extr: 10, cyto: 8, plas: 2, pero: 1 |
| g9287.t1 | cyto | cyto: 15.5, cyto_pero: 10.8333, cyto_nucl: 10.1667, mito: 4.5, extr: 4, pero: 4, nucl: 3.5, E.R._mito: 3 |
| g9374.t1 | nucl | nucl: 32 |
| g9470.t1 | cyto | cyto: 16.5, cyto_nucl: 10, plas: 5, mito: 3, E.R.: 3, nucl: 2.5, extr: 1, golg: 1 |
| g9471.t1 | nucl | nucl: 15.5, cyto_nucl: 10.3333, plas: 8, cyto: 4, cyto_mito: 3.83333, mito: 2.5, lyso: 1, golg: 1 |
| g9621.t1 | mito | mito: 21.5, E.R._mito: 13, nucl: 4, cyto: 3, extr: 1, pero: 1 |
| g9715.t1 | nucl | nucl: 18.5, cyto_nucl: 15.6667, cyto: 9.5, cyto_mito: 6.5, pero: 3 |
| g9719.t1 | nucl | nucl: 10, extr: 9, cyto: 5.5, E.R.: 4, cyto_mito: 3.83333, plas: 2, mito_pero: 1.33333 |
| g9727.t1 | cyto | cyto: 20.5, cyto_nucl: 18.5, nucl: 9.5, mito: 1, lyso: 1 |
| g9728.t1 | nucl | nucl: 18.5, cyto_nucl: 16, cyto: 12.5, mito: 1 |
| g9736.t1 | cyto | cyto: 20.5, cyto_nucl: 12.5, extr: 7, lyso: 2, mito: 1 |
| g9798.t1 | cyto | cyto: 18.5, cyto_nucl: 16, nucl: 12.5, plas: 1 |
| g9823.t1 | nucl | nucl: 13.5, cyto_nucl: 10.5, mito: 5, cyto: 4.5, plas: 4, pero: 3, extr: 1, golg: 1 |
| g9824.t1 | plas | plas: 24, nucl: 5, cyto: 1, mito: 1, pero: 1 |
| g9848.t1 | cyto | cyto: 14, nucl: 11, pero: 5, plas: 1, cysk: 1 |
| g9868.t1 | nucl | nucl: 10.5, cyto_nucl: 8, plas: 5, cyto: 4.5, extr: 4, pero: 4, mito: 3, lyso: 1 |
| g9940.t1 | nucl | nucl: 28, mito: 4 |

Анализ локализации белков с помощью WoLF PSORT показал, что большинство отфильтрованных кандидатов (284 белка) предсказано в ядре (nucl) или в ядерно-цитоплазматическом компартменте (cyto_nucl). Ключевые белки репарации ДНК — MSH6, MLH1, PARP1, ERCC5, PMS2, XPC, а также уникальный белок Dsup — имеют высокие баллы ядерной локализации (28-32), что подтверждает их потенциальную роль в защите геномной ДНК. Часть белков предсказана в митохондриях (DNA полимераза θ, Twinkle хеликаза, MYG1 экзонуклеаза), что может указывать на параллельные механизмы защиты митохондриального генома

Далее воспользуемся инструментом **TargetP-2.0** (тоже является веб-сервером) для предсказания наличия сигнальных мотивов в наших белках-кандидатах. Переходим по ссылке *https://services.healthtech.dtu.dk/service.php?TargetP-2.0*. Выбираем параметры  *Non-plant* и *Short output (no figures)*

Результаты оказались таковы:

| Protein ID | TargetP Prediction | Likelihood (Other) | Likelihood (SP) | Likelihood (mTP) |
|------------|--------------------|--------------------|--------------------|--------------------|
| g10082.t1 | Other | 0.9914 | 0.0005 | 0.0081 |
| g1025.t1 | Other | 0.9357 | 0.0001 | 0.0642 |
| g10297.t1 | Signal peptide | 0.3513 | 0.6482 | 0.0005 |
| g10323.t1 | Other | 0.9937 | 0.0001 | 0.0062 |
| g10325.t1 | Other | 0.9999 | 0.0001 | 0.0001 |
| g10348.t1 | Other | 0.9446 | 0.055 | 0.0005 |
| g10349.t1 | Other | 0.9563 | 0.0005 | 0.0432 |
| g10492.t1 | Other | 0.9967 | 0.0002 | 0.0031 |
| g10574.t1 | Other | 0.8915 | 0.0039 | 0.1046 |
| g10641.t1 | Other | 0.9983 | 0.0005 | 0.0012 |
| g10653.t1 | Other | 0.8262 | 0.0627 | 0.1111 |
| g10676.t1 | Other | 0.9997 | 0 | 0.0002 |
| g10736.t1 | Other | 1 | 0 | 0 |
| g10777.t1 | Other | 0.6711 | 0.0057 | 0.3232 |
| g10796.t1 | Other | 0.9999 | 0 | 0.0001 |
| g10840.t1 | Signal peptide | 0.0368 | 0.9612 | 0.0021 |
| g10855.t1 | Other | 0.9999 | 0.0001 | 0 |
| g10949.t1 | Other | 1 | 0 | 0 |
| g11102.t1 | Other | 0.9999 | 0 | 0.0001 |
| g11171.t1 | Other | 0.9377 | 0.0621 | 0.0002 |
| g11384.t1 | Mitochondrial transfer peptide | 0.4207 | 0.0001 | 0.5792 |
| g11427.t1 | Signal peptide | 0.3471 | 0.6504 | 0.0025 |
| g11432.t1 | Other | 1 | 0 | 0 |
| g11445.t1 | Other | 0.9995 | 0.0001 | 0.0004 |
| g11532.t1 | Other | 0.9999 | 0 | 0.0001 |
| g11557.t1 | Other | 1 | 0 | 0 |
| g11569.t1 | Other | 0.9993 | 0.0006 | 0.0002 |
| g11630.t1 | Other | 1 | 0 | 0 |
| g11632.t1 | Other | 1 | 0 | 0 |
| g11658.t1 | Other | 1 | 0 | 0 |
| g1168.t1 | Other | 0.9545 | 0.0116 | 0.0339 |
| g11715.t1 | Other | 0.997 | 0.0001 | 0.0029 |
| g11721.t1 | Other | 0.998 | 0.0007 | 0.0013 |
| g11724.t1 | Other | 0.9994 | 0.0004 | 0.0002 |
| g11810.t1 | Other | 1 | 0 | 0 |
| g11929.t1 | Other | 0.9996 | 0.0002 | 0.0002 |
| g11960.t1 | Other | 1 | 0 | 0 |
| g12016.t1 | Other | 0.5309 | 0.0542 | 0.4149 |
| g12063.t1 | Other | 1 | 0 | 0 |
| g12121.t1 | Other | 0.9935 | 0.0003 | 0.0062 |
| g12128.t1 | Other | 0.9953 | 0.0002 | 0.0044 |
| g12184.t1 | Other | 0.9996 | 0 | 0.0004 |
| g12193.t1 | Other | 0.9994 | 0.0004 | 0.0001 |
| g12194.t1 | Other | 0.9996 | 0 | 0.0004 |
| g12325.t1 | Other | 0.9997 | 0.0001 | 0.0002 |
| g12416.t1 | Other | 0.9896 | 0.0003 | 0.0101 |
| g1264.t1 | Other | 0.9999 | 0 | 0 |
| g12660.t1 | Other | 0.9976 | 0.0013 | 0.0011 |
| g12742.t1 | Other | 1 | 0 | 0 |
| g12860.t1 | Other | 0.9987 | 0.0009 | 0.0004 |
| g12883.t1 | Other | 0.9948 | 0.0021 | 0.0031 |
| g1292.t1 | Other | 0.9999 | 0 | 0.0001 |
| g12972.t1 | Other | 1 | 0 | 0 |
| g13018.t1 | Other | 1 | 0 | 0 |
| g13042.t1 | Other | 0.9998 | 0.0001 | 0.0001 |
| g13117.t1 | Other | 0.9999 | 0.0001 | 0 |
| g13177.t1 | Other | 0.9998 | 0 | 0.0002 |
| g13269.t1 | Other | 1 | 0 | 0 |
| g13403.t1 | Other | 0.9983 | 0.0003 | 0.0014 |
| g13489.t1 | Other | 1 | 0 | 0 |
| g13540.t1 | Other | 0.9981 | 0.0001 | 0.0018 |
| g13571.t1 | Other | 0.9999 | 0 | 0.0001 |
| g13635.t1 | Other | 0.8215 | 0.0101 | 0.1684 |
| g13696.t1 | Other | 0.9755 | 0.0007 | 0.0238 |
| g13775.t1 | Other | 1 | 0 | 0 |
| g13868.t1 | Other | 0.997 | 0.003 | 0 |
| g13911.t1 | Other | 0.9997 | 0 | 0.0002 |
| g14015.t1 | Other | 0.9928 | 0 | 0.0072 |
| g14206.t1 | Other | 0.9999 | 0 | 0 |
| g14252.t1 | Other | 0.9999 | 0 | 0.0001 |
| g14271.t1 | Other | 1 | 0 | 0 |
| g14279.t1 | Other | 0.9997 | 0 | 0.0002 |
| g14353.t1 | Other | 0.9994 | 0.0004 | 0.0002 |
| g14472.t1 | Other | 1 | 0 | 0 |
| g14479.t1 | Other | 0.9996 | 0.0004 | 0 |
| g14532.t1 | Other | 0.9991 | 0 | 0.0009 |
| g14703.t1 | Other | 0.9964 | 0.0034 | 0.0002 |
| g14739.t1 | Other | 1 | 0 | 0 |
| g14782.t1 | Other | 0.8951 | 0.0002 | 0.1047 |
| g14826.t1 | Other | 0.9997 | 0.0001 | 0.0002 |
| g14827.t1 | Other | 0.9991 | 0.0001 | 0.0007 |
| g14891.t1 | Other | 0.9986 | 0.0002 | 0.0012 |
| g14936.t1 | Other | 0.9822 | 0.007 | 0.0108 |
| g14959.t1 | Other | 1 | 0 | 0 |
| g14969.t1 | Other | 0.9999 | 0 | 0.0001 |
| g14980.t1 | Other | 0.6869 | 0.312 | 0.0011 |
| g14998.t1 | Other | 0.9989 | 0.0001 | 0.001 |
| g15015.t1 | Other | 0.9952 | 0.0045 | 0.0002 |
| g15048.t1 | Other | 1 | 0 | 0 |
| g15127.t1 | Other | 0.9993 | 0.0002 | 0.0005 |
| g15275.t1 | Other | 0.9998 | 0.0001 | 0.0001 |
| g15443.t1 | Other | 0.9996 | 0.0002 | 0.0002 |
| g15466.t1 | Other | 1 | 0 | 0 |
| g15521.t1 | Other | 0.9999 | 0 | 0.0001 |
| g15558.t1 | Signal peptide | 0.0053 | 0.9945 | 0.0001 |
| g15649.t1 | Other | 0.9998 | 0.0001 | 0 |
| g15676.t1 | Other | 0.9993 | 0.0005 | 0.0002 |
| g15760.t1 | Other | 0.9408 | 0.0582 | 0.001 |
| g15808.t1 | Other | 1 | 0 | 0 |
| g15812.t1 | Other | 0.9998 | 0.0001 | 0.0001 |
| g15826.t1 | Other | 0.9999 | 0 | 0.0001 |
| g15867.t1 | Other | 0.9974 | 0.0005 | 0.0021 |
| g16064.t1 | Other | 0.9999 | 0 | 0.0001 |
| g16168.t1 | Other | 0.9975 | 0 | 0.0025 |
| g162.t1 | Mitochondrial transfer peptide | 0.4342 | 0.0015 | 0.5644 |
| g16314.t1 | Other | 0.9988 | 0.0011 | 0.0001 |
| g16404.t1 | Other | 0.9999 | 0 | 0.0001 |
| g16412.t1 | Other | 0.9999 | 0 | 0.0001 |
| g16416.t1 | Other | 0.9999 | 0 | 0.0001 |
| g16417.t1 | Other | 0.9999 | 0 | 0.0001 |
| g1662.t1 | Other | 0.9922 | 0.0004 | 0.0074 |
| g1709.t1 | Other | 0.9864 | 0.0131 | 0.0005 |
| g1742.t1 | Other | 0.9989 | 0.0005 | 0.0006 |
| g1774.t1 | Mitochondrial transfer peptide | 0.311 | 0.0005 | 0.6885 |
| g1818.t1 | Other | 0.9999 | 0 | 0.0001 |
| g1871.t1 | Other | 0.9996 | 0.0002 | 0.0002 |
| g189.t1 | Other | 1 | 0 | 0 |
| g1892.t1 | Other | 0.9999 | 0 | 0.0001 |
| g1895.t1 | Mitochondrial transfer peptide | 0.2796 | 0.0003 | 0.7202 |
| g2016.t1 | Other | 0.9668 | 0.0329 | 0.0003 |
| g2038.t1 | Other | 0.9995 | 0 | 0.0005 |
| g2056.t1 | Other | 0.9876 | 0.0043 | 0.008 |
| g2074.t1 | Other | 0.9997 | 0.0002 | 0.0001 |
| g2168.t1 | Other | 1 | 0 | 0 |
| g2278.t1 | Signal peptide | 0.0128 | 0.8357 | 0.1515 |
| g2331.t1 | Other | 1 | 0 | 0 |
| g2444.t1 | Other | 0.9976 | 0.0003 | 0.0021 |
| g2580.t1 | Other | 1 | 0 | 0 |
| g2617.t1 | Other | 0.9999 | 0.0001 | 0 |
| g2755.t1 | Other | 1 | 0 | 0 |
| g2756.t1 | Other | 0.9746 | 0.0119 | 0.0136 |
| g2760.t1 | Other | 0.9871 | 0.0011 | 0.0117 |
| g2873.t1 | Other | 0.9999 | 0 | 0.0001 |
| g305.t1 | Other | 0.9999 | 0 | 0 |
| g3355.t1 | Other | 1 | 0 | 0 |
| g340.t1 | Other | 1 | 0 | 0 |
| g3515.t1 | Mitochondrial transfer peptide | 0.4693 | 0.0004 | 0.5304 |
| g3540.t1 | Other | 0.9164 | 0.0803 | 0.0033 |
| g356.t1 | Other | 0.9998 | 0.0002 | 0 |
| g3566.t1 | Other | 1 | 0 | 0 |
| g3636.t1 | Other | 0.8209 | 0.179 | 0.0001 |
| g3737.t1 | Other | 1 | 0 | 0 |
| g3813.t1 | Other | 0.9994 | 0.0003 | 0.0003 |
| g391.t1 | Other | 0.5638 | 0.0002 | 0.436 |
| g3932.t1 | Other | 0.9995 | 0.0003 | 0.0002 |
| g3982.t1 | Other | 1 | 0 | 0 |
| g4002.t1 | Other | 0.8445 | 0.0127 | 0.1428 |
| g4025.t1 | Other | 0.9996 | 0.0001 | 0.0003 |
| g4032.t1 | Other | 0.997 | 0.0006 | 0.0024 |
| g4099.t1 | Mitochondrial transfer peptide | 0.3041 | 0.0027 | 0.6932 |
| g410.t1 | Other | 0.9842 | 0.0156 | 0.0002 |
| g411.t1 | Other | 0.9998 | 0 | 0.0002 |
| g4160.t1 | Other | 0.9999 | 0 | 0.0001 |
| g4200.t1 | Other | 0.9956 | 0 | 0.0044 |
| g4201.t1 | Other | 0.9998 | 0.0001 | 0.0001 |
| g4324.t1 | Other | 0.9999 | 0 | 0 |
| g4327.t1 | Other | 0.9968 | 0 | 0.0032 |
| g4349.t1 | Other | 0.995 | 0.0046 | 0.0004 |
| g4361.t1 | Other | 0.9996 | 0.0001 | 0.0003 |
| g4504.t1 | Other | 1 | 0 | 0 |
| g4520.t1 | Other | 0.996 | 0.0005 | 0.0035 |
| g4570.t1 | Other | 1 | 0 | 0 |
| g4585.t1 | Other | 1 | 0 | 0 |
| g4593.t1 | Other | 1 | 0 | 0 |
| g4613.t1 | Other | 0.9999 | 0.0001 | 0 |
| g4634.t1 | Other | 0.9999 | 0.0001 | 0 |
| g4704.t1 | Other | 0.9996 | 0.0001 | 0.0002 |
| g4738.t1 | Other | 0.9879 | 0.005 | 0.0071 |
| g4740.t1 | Other | 0.9988 | 0 | 0.0012 |
| g4789.t1 | Other | 0.996 | 0.0037 | 0.0003 |
| g4851.t1 | Other | 0.9714 | 0.0001 | 0.0285 |
| g4890.t1 | Other | 0.9986 | 0.0004 | 0.001 |
| g494.t1 | Other | 0.9639 | 0.0006 | 0.0355 |
| g5065.t1 | Other | 0.9994 | 0.0001 | 0.0005 |
| g5069.t1 | Other | 0.9984 | 0.0001 | 0.0014 |
| g5104.t1 | Other | 0.9945 | 0.0003 | 0.0052 |
| g518.t1 | Other | 0.6961 | 0.0093 | 0.2946 |
| g5302.t1 | Other | 0.9974 | 0.0003 | 0.0023 |
| g5337.t1 | Other | 0.9998 | 0.0001 | 0.0001 |
| g5381.t1 | Other | 1 | 0 | 0 |
| g5456.t1 | Other | 0.9999 | 0 | 0.0001 |
| g5582.t1 | Other | 0.9999 | 0 | 0 |
| g5618.t1 | Other | 0.9996 | 0.0004 | 0 |
| g5721.t1 | Other | 0.9997 | 0 | 0.0003 |
| g5905.t1 | Other | 0.9963 | 0.0005 | 0.0032 |
| g5927.t1 | Other | 1 | 0 | 0 |
| g5942.t1 | Other | 0.8281 | 0.0059 | 0.166 |
| g5968.t1 | Other | 1 | 0 | 0 |
| g5997.t1 | Other | 0.9915 | 0.0004 | 0.0081 |
| g6016.t1 | Mitochondrial transfer peptide | 0.4729 | 0.0017 | 0.5255 |
| g6080.t1 | Other | 1 | 0 | 0 |
| g6081.t1 | Other | 0.9992 | 0.0005 | 0.0003 |
| g6093.t1 | Other | 1 | 0 | 0 |
| g6118.t1 | Other | 1 | 0 | 0 |
| g6125.t1 | Other | 1 | 0 | 0 |
| g623.t1 | Other | 1 | 0 | 0 |
| g6239.t1 | Mitochondrial transfer peptide | 0.4231 | 0.0003 | 0.5766 |
| g6345.t1 | Other | 0.9992 | 0.0005 | 0.0003 |
| g6351.t1 | Other | 1 | 0 | 0 |
| g6372.t1 | Other | 0.9727 | 0.0251 | 0.0022 |
| g6459.t1 | Other | 0.9795 | 0.0002 | 0.0204 |
| g6472.t1 | Other | 0.9999 | 0 | 0 |
| g651.t1 | Other | 0.9979 | 0.0003 | 0.0018 |
| g6593.t1 | Other | 0.9999 | 0 | 0.0001 |
| g6605.t1 | Other | 0.9989 | 0.001 | 0 |
| g6630.t1 | Other | 0.9998 | 0.0002 | 0 |
| g6657.t1 | Other | 1 | 0 | 0 |
| g6694.t1 | Other | 1 | 0 | 0 |
| g6698.t1 | Other | 0.9575 | 0.0104 | 0.0321 |
| g673.t1 | Other | 0.9998 | 0.0002 | 0 |
| g6744.t1 | Other | 0.9999 | 0 | 0 |
| g6791.t1 | Other | 0.9994 | 0.0002 | 0.0004 |
| g691.t1 | Other | 1 | 0 | 0 |
| g6979.t1 | Other | 1 | 0 | 0 |
| g7087.t1 | Other | 0.9952 | 0.0043 | 0.0006 |
| g709.t1 | Other | 0.5377 | 0.4566 | 0.0056 |
| g7179.t1 | Other | 0.9993 | 0.0004 | 0.0003 |
| g7201.t1 | Other | 0.7374 | 0.0004 | 0.2622 |
| g7211.t1 | Other | 0.9996 | 0.0001 | 0.0003 |
| g7224.t1 | Other | 1 | 0 | 0 |
| g7260.t1 | Other | 0.8789 | 0.0054 | 0.1157 |
| g7261.t1 | Other | 0.9987 | 0.0002 | 0.0011 |
| g7303.t1 | Other | 1 | 0 | 0 |
| g7338.t1 | Other | 0.8683 | 0.0002 | 0.1315 |
| g7355.t1 | Other | 1 | 0 | 0 |
| g737.t1 | Other | 0.9999 | 0 | 0.0001 |
| g7377.t1 | Other | 1 | 0 | 0 |
| g7394.t1 | Other | 1 | 0 | 0 |
| g7454.t1 | Other | 0.9999 | 0 | 0 |
| g7471.t1 | Other | 0.998 | 0.0001 | 0.0019 |
| g7488.t1 | Other | 0.9999 | 0 | 0.0001 |
| g7517.t1 | Other | 0.9997 | 0.0001 | 0.0002 |
| g760.t1 | Other | 1 | 0 | 0 |
| g7637.t1 | Other | 0.9685 | 0.0009 | 0.0306 |
| g7744.t1 | Other | 0.9999 | 0.0001 | 0 |
| g7761.t1 | Other | 0.9999 | 0 | 0.0001 |
| g7770.t1 | Other | 1 | 0 | 0 |
| g7861.t1 | Other | 1 | 0 | 0 |
| g7928.t1 | Other | 1 | 0 | 0 |
| g7935.t1 | Other | 0.8062 | 0.1936 | 0.0003 |
| g7961.t1 | Other | 0.9996 | 0 | 0.0004 |
| g7998.t1 | Other | 1 | 0 | 0 |
| g8013.t1 | Other | 1 | 0 | 0 |
| g8026.t1 | Other | 0.8532 | 0.0042 | 0.1426 |
| g8065.t1 | Other | 0.9998 | 0.0002 | 0 |
| g809.t1 | Other | 0.9242 | 0.0001 | 0.0757 |
| g8130.t1 | Other | 0.9998 | 0.0001 | 0 |
| g8150.t1 | Other | 0.9978 | 0 | 0.0021 |
| g832.t1 | Other | 0.9616 | 0.0379 | 0.0005 |
| g833.t1 | Other | 0.9929 | 0.0019 | 0.0052 |
| g8350.t1 | Other | 1 | 0 | 0 |
| g8387.t1 | Other | 0.9991 | 0.0005 | 0.0004 |
| g849.t1 | Other | 0.8897 | 0.0005 | 0.1099 |
| g8593.t1 | Other | 0.9541 | 0.0095 | 0.0364 |
| g8602.t1 | Other | 1 | 0 | 0 |
| g8658.t1 | Other | 1 | 0 | 0 |
| g8660.t1 | Other | 0.9976 | 0.0002 | 0.0021 |
| g8742.t1 | Other | 1 | 0 | 0 |
| g8771.t1 | Other | 1 | 0 | 0 |
| g8850.t1 | Other | 1 | 0 | 0 |
| g8854.t1 | Other | 0.9981 | 0.0004 | 0.0015 |
| g8882.t1 | Other | 0.9972 | 0 | 0.0028 |
| g8905.t1 | Other | 1 | 0 | 0 |
| g8953.t1 | Other | 0.9589 | 0.0002 | 0.0409 |
| g9035.t1 | Other | 0.9999 | 0 | 0 |
| g9081.t1 | Other | 0.9999 | 0 | 0.0001 |
| g9097.t1 | Other | 0.9886 | 0.0074 | 0.004 |
| g9168.t1 | Other | 0.9783 | 0.001 | 0.0207 |
| g9287.t1 | Other | 0.99 | 0.0096 | 0.0004 |
| g9374.t1 | Other | 0.8857 | 0.1135 | 0.0008 |
| g9470.t1 | Other | 0.9934 | 0.0014 | 0.0052 |
| g9471.t1 | Other | 0.9999 | 0.0001 | 0 |
| g9621.t1 | Other | 0.9999 | 0 | 0.0001 |
| g9715.t1 | Other | 0.9999 | 0.0001 | 0 |
| g9719.t1 | Other | 0.9516 | 0.0023 | 0.0461 |
| g9727.t1 | Other | 0.9981 | 0.0003 | 0.0016 |
| g9728.t1 | Other | 0.9998 | 0.0002 | 0 |
| g9736.t1 | Other | 0.9975 | 0.0006 | 0.0019 |
| g9798.t1 | Other | 0.9997 | 0.0001 | 0.0002 |
| g9823.t1 | Other | 0.726 | 0.0117 | 0.2623 |
| g9824.t1 | Other | 0.9978 | 0.0008 | 0.0014 |
| g9848.t1 | Other | 1 | 0 | 0 |
| g9868.t1 | Other | 0.9974 | 0.0023 | 0.0003 |
| g9940.t1 | Other | 0.9998 | 0.0001 | 0.0001 |

Анализ результатов TargetP-2.0 показал, что подавляющее большинство отобранных белков-кандидатов имеют предсказание **«Other»** с очень высокой вероятностью. Это означает, что в их N-концевых последовательностях отсутствуют сигнальные пептиды, направляющие белок в секреторный путь (Signal Peptide, SP) или митохондрии (mitochondrial transit peptide, mTP). Данный результат логично согласуется с данными WoLF PSORT, где эти же белки были преимущественно отнесены к ядерной (nucl) или цитоплазматической (cyto) локализации. Таким образом, для ключевых кандидатов - Dsup, PARP1, MSH6, MLH1, DNA-полимераз, хеликаз и гистонметилтрансфераз - оба метода предсказания локализации дают согласованные результаты. Лишь несколько белков (например, g10297.t1, g10840.t1, g15558.t1) были предсказаны как имеющие сигнальный пептид (SP) и, следовательно, направляющиеся в секреторную систему. Также у небольшой части белков (g11384.t1, g162.t1, g1895.t1, g3515.t1, g4099.t1, g6016.t1) TargetP предсказал митохондриальный транзитный пептид (mTP), что делает их кандидатами на роль защитников митохондриального генома

8. Переходим к предсказанию наличия консервативных доменов в белках, используя **HMMER**

Для этого перенесём 75 наиболее интересующих нас белковых последовательностей из 284 в отдельный файл для удобства дальнейшего анализа

В файл top_proteins_for_pfam.txt перенесём индексы нужных белков

Извлечём эти белки с помощью seqtk

In [ ]:
seqtk subseq filtered_candidates.faa top_proteins_to_pfam.txt > top_proteins_to_pfam.faa

Проверим, все ли 75 белков извлеклись

In [ ]:
grep -c "^>" top_proteins_to_pfam.faa

75

Всё успешно извлеклось. Переходим теперь непосредственно к поиску мотивов. Переходим по ссылке *https://www.ebi.ac.uk/Tools/hmmer/*. Выбираем вкладку *Search*, затем параметры *hmmscan*, cut off *Gathering*

Наиболее интересные и значимые результаты (из всез 75 последовательностей отобраны лишь 18, так как, на наш взгляд, являются лучшими хитами) представлены в таблице ниже. Также в итоговой таблице для сравнения представлены столбцы, соответствующие результатам WoLF PSORT и TargetP

| Protein ID | Best BLAST Hit (E-value, %Ident) | Pfam Domains (E-value) | WoLF PSORT | TargetP |
|------------|----------------------------------|------------------------|------------|---------|
| g14472.t1 | Damage suppressor protein (0.0, 100%) | (no Pfam hits - unique) | nucl | Other |
| g12184.t1 | DNA mismatch repair protein Msh6 (0.0, 41%) | MutS_I (1.2e-30), MutS_III (2.3e-25) | nucl | Other |
| g4585.t1 | Poly [ADP-ribose] polymerase 1 (6.5e-177, 36%) | PARP (2.5e-76), WGR (2.3e-16), zf-PARP (3.2e-14) | nucl | Other |
| g9374.t1 | DNA mismatch repair protein Mlh1 (0.0, 42%) | Mlh1_C (3.0e-63), DNA_mis_repair (1.1e-30) | nucl | Other |
| g1025.t1 | DNA polymerase theta (0.0, 47%) | POLQ_helical (5.9e-51), HTH_61 (6.3e-24) | mito | Other |
| g518.t1 | DNA excision repair protein ERCC-6 (0.0, 58.6%) | SNF2-rel_dom (2.4e-71), Helicase_C (2.0e-25) | nucl | Other |
| g7998.t1 | ERCC3 (0.0, 69.4%) | ERCC3_C (2.4e-111), Helicase_C (2.5e-09) | nucl | Other |
| g11445.t1 | DNA excision repair protein ERCC-5 (2.4e-75, 40%) | XPG_N (3.1e-29), XPG_I (3.0e-24) | nucl | Other |
| g11810.t1 | Double-strand break repair protein MRE11 (1.8e-99, 44%) | Mre11_DNA_bind (2.4e-30), Metallophos (3.1e-06) | cyto_nucl | Other |
| g7211.t1 | Mismatch repair endonuclease PMS2 (2.5e-56, 36.8%) | HATPase_c_3 (1.5e-10), MutL_C (1.3e-09) | nucl | Other |
| g4002.t1 | DNA topoisomerase 3-beta-1 (0.0, 49.2%) | Topoisom_bac (3.0e-99), Toprim (5.5e-12) | cyto | Other |
| g13571.t1 | ATP-dependent DNA/RNA helicase DHX36 (0.0, 43.1%) | DEAD (3.2e-13), Helicase_C (3.4e-15), HA2 (3.2e-11) | nucl | Other |
| g11102.t1 | Probable ATP-dependent RNA helicase DHX35 (0.0, 57.3%) | OB_NTP_bind (2.3e-23), HA2 (2.2e-19), Helicase_C (8.6e-15) | nucl | Other |
| g15443.t1 | Histone-binding protein RBBP4 (0.0, 67.7%) | CAF1C_H4-bd (7.2e-29), WD40 (multiple) | cysk | Other |
| g8593.t1 | Histone H2AX (1.4e-47, 64.8%) | Histone_H2A_C (5.0e-15), Histone (2.3e-07) | plas | Other |
| g14827.t1 | Heat shock protein 105 kDa (0.0, 41.8%) | HSP70 (multiple) | nucl | Other |
| g1818.t1 | DnaJ homolog subfamily C member 2 (3.6e-140, 42.6%) | DnaJ (2.4e-16), Myb_DNA-binding (2.4e-08) | nucl | Other |

Отбор представленных в таблице белков проводился на основе интеграции нескольких строгих критериев, направленных на выявление наиболее надежных и функционально важных кандидатов, потенциально участвующих в защите ДНК у *Ramazzottius varieornatus*.

- **Первый и главный критерий** - это результат гомологического поиска (BLAST) против базы Swiss-Prot. В таблицу включались только белки с крайне низкими E-value, преимущественно равными 0.0, что свидетельствует о практически абсолютной достоверности гомологии. Исключение составил белок ERCC-5 (g11445.t1) с E-value 2.4e-75, который также был включен из-за его критически важной роли в эксцизионной репарации нуклеотидов. Процент идентичности варьировал от 36% до 100%, причем уникальный белок Dsup (g14472.t1) показал 100% идентичность самому себе, что подтверждает его специфичность для тихоходок
  
- **Вторым ключевым критерием** был анализ консервативных доменов с помощью HMMER. Для каждого кандидата были идентифицированы домены, соответствующие его предполагаемой функции. Например, у белка MSH6 (g12184.t1) обнаружены домены MutS_I и MutS_III, характерные для белков системы репарации неспаренных оснований. У PARP1 (g4585.t1) присутствуют каталитический PARP-домен, а также WGR и цинк-пальцевый домены, необходимые для узнавания повреждений ДНК. Наличие доменов SNF2-rel_dom и Helicase_C у ERCC6 (g518.t1) и ERCC3 (g7998.t1) подтверждает их хеликазную активность, критичную для эксцизионной репарации. Особого внимания заслуживает белок Dsup, для которого не было найдено ни одного домена в базе Pfam, что подчеркивает его уникальную, вероятно, эволюционно новую структуру, не встречающуюся у других организмов

- **Третий критерий** - предсказание субклеточной локализации с помощью WoLF PSORT и TargetP. Приоритет отдавался белкам с предсказанием в ядре (nucl) или в ядерно-цитоплазматическом компартменте (cyto_nucl) по данным WoLF PSORT, а также с отсутствием N-концевых сигнальных пептидов по данным TargetP (метка Other). Это условие критически важно, поскольку белки, ассоциированные с хроматином и защищающие ядерную ДНК, должны локализоваться именно в ядре. Все отобранные кандидаты, за исключением белка NEDD4 (g11427.t1, который был исключен из-за локализации в ЭПР и наличия сигнального пептида), соответствуют этому критерию. Белки DNA polymerase theta (g1025.t1) и RBBP4 (g15443.t1) были включены, несмотря на предсказание в митохондриях и цитоскелете, из-за их несомненной важности (полимераза тета уникальна для репарации двухцепочечных разрывов, а RBBP4 - ключевой шаперон гистонов). Histone H2AX (g8593.t1) был оставлен в списке, несмотря на предсказание в плазматической мембране, так как его роль как маркера повреждений ДНК общеизвестна и подтверждена Pfam доменами гистонов

- **Четвертый критерий** - биологическая правдоподобность и релевантность гипотезе. В таблицу включались белки, чьи функции напрямую связаны с репарацией ДНК (MSH6, MLH1, PMS2, MRE11, ERCC5, ERCC6, ERCC3, DNA полимераза тета), поддержанием структуры хроматина (гистоны H2AX, RBBP4), регуляцией стрессового ответа (белки теплового шока HSP105 и DnaJ) или деградацией поврежденных белков (убиквитин-лигаза NEDD4, которая, однако, не прошла локализационный фильтр). Таким образом, финальный список представляет собой сбалансированный набор высоконадежных кандидатов, охватывающих различные механизмы потенциальной защиты генома

9. **ВЫВОД**:

На основании данных масс-спектрометрии, гомологического поиска (BLAST), предсказания субклеточной локализации (WoLF PSORT, TargetP) и анализа консервативных доменов (Pfam) из 75 белков были отобраны 18 приоритетных кандидатов для экспериментальной проверки. Ключевыми критериями отбора являлись экстремально низкие E-value (преимущественно 0.0), ядерная локализация (nucl/cyto_nucl) и отсутствие сигнальных пептидов (TargetP: Other), а также наличие консервативных доменов, соответствующих функции. Наиболее перспективными кандидатами признаны уникальный белок Dsup (100% идентичность, ядерная локализация), классические факторы репарации ДНК (MSH6, MLH1, PARP1, PMS2, MRE11, ERCC5, ERCC6, ERCC3, DNA полимераза тета), ДНК-хеликазы (TOP3B, DHX36), гистоны (H2AX, RBBP4) и белки стрессового ответа (HSP105, DnaJ). Полученные результаты подтверждают гипотезу о том, что тихоходки используют комбинацию уникальных (Dsup) и высококонсервативных механизмов для защиты геномной ДНК от повреждений, и предоставляют конкретный список мишеней для дальнейших экспериментов по валидации.